# Solvi — fully self-contained pipeline (zero `solvi` imports)

This notebook reproduces the **entire** Solvi LangGraph pipeline **inline** — every
node, prompt, schema, HTML template, and helper. It imports only third-party
packages (pydantic, litellm, langgraph, jinja2, sympy, optional rdkit); there is
**no `import solvi` / `import llm_gateway` anywhere**.

**All 21 nodes are here:** ingest → classify → kc_tag → solve ⇄ verify → score →
route → {manim | rdkit | mol3d | matter | canvas | text | withhold} → (manim:
narration → frame_check → mux) → finalize.

**Two things that make it run:**

- **`viz_type` + `smiles` are restored** on the inline `Classification` (the live
  repo dropped them, which breaks routing) — so the graph routes end-to-end.
- **Honest degradation:** every generator/audio node wraps its body in
  try/except and falls back to the text artifact. So even without LaTeX / ffmpeg
  / manim / OpenAI-TTS the pipeline completes — a run just traverses the nodes
  its routing reaches, degrading the rest.

**Real LLM calls.** Loads `.env`; default model `gemini` (`gemini-2.0-flash`).
Gemini free tier 429s under load — set `MODELS={"default":"nv-llama3.1-8b"}` in
the config cell to use the fast NVIDIA NIM fallback.

Run the cells top to bottom. Edit the **config** and **question** cells to steer
which engine the router picks (see the routing presets in the run cell).

## 1 · Setup — environment, keys, paths

In [ ]:
# === Setup: environment, keys, paths ===
# This notebook is SELF-CONTAINED: it does NOT import `solvi` or `llm_gateway`.
# Every node, prompt, schema, and template is reproduced inline below.
import os
import sys
from pathlib import Path
from types import SimpleNamespace

from dotenv import load_dotenv

# Where to find the JEE syllabus JSON (data file, not solvi code). Point this
# anywhere; defaults to the repo copy. Only the kc_tag node needs it.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "syallbus_data").is_dir():
    REPO_ROOT = REPO_ROOT.parent          # running from notebooks/
SYLLABUS_PATH = Path(
    os.getenv("SOLVI_SYLLABUS_PATH",
              REPO_ROOT / "syallbus_data" / "jee_main_2025_syllabus_atomic.json")
)

# Scratch dir for any rendered artifacts (manim mp4s, audio). HTML artifacts are
# returned as strings and never touch disk.
ARTIFACT_DIR = Path(os.getenv("SOLVI_ARTIFACT_DIR", REPO_ROOT / "var" / "artifacts"))

load_dotenv(REPO_ROOT / ".env")

# Process limits (mirror solvi.config.SolviSettings).
SETTINGS = SimpleNamespace(
    artifact_dir=ARTIFACT_DIR,
    syllabus_path=SYLLABUS_PATH,
    safe_exec_timeout=float(os.getenv("SOLVI_SAFE_EXEC_TIMEOUT", "10")),
    render_timeout=float(os.getenv("SOLVI_RENDER_TIMEOUT", "600")),
    max_image_bytes=int(os.getenv("SOLVI_MAX_IMAGE_BYTES", str(10 * 1024 * 1024))),
)


def settings():
    """Stand-in for solvi.config.settings() — process limits only."""
    return SETTINGS


for var in ("GEMINI_API_KEY", "NVIDIA_NIM_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    print(f"{var:20s} {'set' if os.getenv(var) else 'MISSING'}")
print("syllabus:", SYLLABUS_PATH, "exists" if SYLLABUS_PATH.exists() else "MISSING")
print("no solvi import:", "solvi" not in sys.modules)


## 2 · LLM layer (inlined gateway: alias table + parse/retry)

In [ ]:
# === LLM layer (inlined llm_gateway) ===
# Mirrors llm_gateway/{config,structured,gateway,multimodal}.py: a small alias
# table, schema-as-system-prompt + parse-and-retry (NO json-mode — NIM stalls on
# it), and text/speech/transcribe entry points. Exposed as an `llm` namespace so
# the node bodies below read exactly like the real solvi code.
import json
import litellm
from pydantic import BaseModel, ValidationError

litellm.drop_params = True   # silently drop unsupported params instead of erroring

# --- alias table (subset of llm_gateway/models.yaml) ---
_NIM_BASE = os.getenv("NVIDIA_NIM_API_BASE", "https://integrate.api.nvidia.com/v1")
MODEL_REGISTRY = {
    "gemini":         dict(model="gemini/gemini-2.0-flash", api_key_env="GEMINI_API_KEY"),
    "nv-llama3.1-8b": dict(model="nvidia_nim/meta/llama-3.1-8b-instruct",
                           api_key_env="NVIDIA_NIM_API_KEY", api_base=_NIM_BASE, max_tokens=4096),
    "nv-llama3.3-70b": dict(model="nvidia_nim/meta/llama-3.3-70b-instruct",
                            api_key_env="NVIDIA_NIM_API_KEY", api_base=_NIM_BASE, max_tokens=4096),
    "nv-llama3.2-90b-vision": dict(model="nvidia_nim/meta/llama-3.2-90b-vision-instruct",
                                   api_key_env="NVIDIA_NIM_API_KEY", api_base=_NIM_BASE, max_tokens=4096),
    "tts": dict(model="openai/gpt-4o-mini-tts", api_key_env="OPENAI_API_KEY"),
    "stt": dict(model="openai/gpt-4o-mini-transcribe", api_key_env="OPENAI_API_KEY"),
}
GATEWAY_NUM_RETRIES = 2
GATEWAY_TIMEOUT = 60.0


def resolve_alias(alias):
    """alias -> config dict. Unknown alias is treated as a raw provider/model."""
    if alias in MODEL_REGISTRY:
        return MODEL_REGISTRY[alias]
    return {"model": alias}


# --- multimodal (llm_gateway/multimodal.py) ---
_URL_PREFIXES = ("http://", "https://", "data:")


def image_content(image, mime_type="image/png"):
    if isinstance(image, dict):
        kind, data = image.get("kind"), image.get("data", "")
        mime = image.get("mime_type", mime_type)
        if kind == "url":
            url = data
        elif kind == "base64":
            url = f"data:{mime};base64,{data}"
        else:
            raise ValueError(f"image dict 'kind' must be 'url' or 'base64', got {kind!r}")
    elif isinstance(image, str):
        url = image if image.startswith(_URL_PREFIXES) else f"data:{mime_type};base64,{image}"
    else:
        raise TypeError(f"unsupported image type: {type(image).__name__}")
    if not url:
        raise ValueError("image reference resolved to an empty URL")
    return {"type": "image_url", "image_url": {"url": url}}


def build_user_content(text, images):
    if not images:
        return text
    parts = [{"type": "text", "text": text}]
    parts.extend(image_content(img) for img in images)
    return parts


def _completion_kwargs(alias, messages, **overrides):
    cfg = resolve_alias(alias)
    kw = dict(model=cfg["model"], messages=messages,
              temperature=cfg.get("temperature", 0.7),
              num_retries=GATEWAY_NUM_RETRIES, timeout=GATEWAY_TIMEOUT)
    if cfg.get("max_tokens") is not None:
        kw["max_tokens"] = cfg["max_tokens"]
    if cfg.get("api_base"):
        kw["api_base"] = cfg["api_base"]
    if cfg.get("api_key_env"):
        key = os.getenv(cfg["api_key_env"])
        if key:
            kw["api_key"] = key
    kw.update(overrides)
    return kw


def complete_text(alias, prompt, system=None, images=None, **overrides):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": build_user_content(prompt, images)})
    resp = litellm.completion(**_completion_kwargs(alias, messages, **overrides))
    return resp["choices"][0]["message"]["content"] or ""


# --- structured (llm_gateway/structured.py): schema-in-prompt + parse/retry ---
_SCHEMA_INSTRUCTION = """\
Respond with ONLY a single JSON object that conforms to this JSON Schema.
No prose before or after it, no markdown fences.

JSON Schema:
{schema}
"""
_RETRY_INSTRUCTION = """\
Your previous reply was not a valid response:
{error}

Reply again with ONLY a single valid JSON object matching the schema. \
No prose, no markdown fences."""


def extract_json(text):
    t = text.strip()
    if t.startswith("```"):
        nl = t.find("\n")
        if nl != -1:
            t = t[nl + 1:]
        if t.rstrip().endswith("```"):
            t = t.rstrip()[:-3]
    start, end = t.find("{"), t.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("no JSON object found in model output")
    return t[start:end + 1]


def complete_structured(alias, prompt, schema, system=None, images=None, max_attempts=3, **overrides):
    schema_json = json.dumps(schema.model_json_schema(), indent=2)
    sys_content = _SCHEMA_INSTRUCTION.format(schema=schema_json)
    if system:
        sys_content = f"{system}\n\n{sys_content}"
    messages = [{"role": "system", "content": sys_content},
                {"role": "user", "content": build_user_content(prompt, images)}]
    raw, last = "", None
    for _ in range(max_attempts):
        resp = litellm.completion(**_completion_kwargs(alias, messages, **overrides))
        raw = resp["choices"][0]["message"]["content"] or ""
        try:
            return schema.model_validate_json(extract_json(raw))
        except (ValidationError, ValueError, json.JSONDecodeError) as e:
            last = e
            messages = messages + [
                {"role": "assistant", "content": raw},
                {"role": "user", "content": _RETRY_INSTRUCTION.format(error=e)},
            ]
    raise RuntimeError(f"model failed to produce valid {schema.__name__} JSON "
                       f"after {max_attempts} attempts: {last}")


def complete_speech(alias, text, voice="coral", instructions=None, response_format="mp3", **ov):
    cfg = resolve_alias(alias)
    kw = dict(model=cfg["model"], input=text, voice=voice,
              response_format=response_format, num_retries=GATEWAY_NUM_RETRIES,
              timeout=GATEWAY_TIMEOUT)
    if instructions:
        kw["instructions"] = instructions
    if cfg.get("api_key_env") and os.getenv(cfg["api_key_env"]):
        kw["api_key"] = os.getenv(cfg["api_key_env"])
    kw.update(ov)
    resp = litellm.speech(**kw)
    for attr in ("read", "content"):
        if hasattr(resp, attr):
            v = getattr(resp, attr)
            return v() if callable(v) else v
    return bytes(resp)


def complete_transcribe(alias, file_path, language=None, **ov):
    cfg = resolve_alias(alias)
    kw = dict(model=cfg["model"], num_retries=GATEWAY_NUM_RETRIES, timeout=GATEWAY_TIMEOUT)
    if language:
        kw["language"] = language
    if cfg.get("api_key_env") and os.getenv(cfg["api_key_env"]):
        kw["api_key"] = os.getenv(cfg["api_key_env"])
    kw.update(ov)
    with open(file_path, "rb") as fh:
        resp = litellm.transcription(file=fh, **kw)
    return getattr(resp, "text", "") or ""


# --- per-node model resolution (solvi/llm.py) ---
def _model_for(state, node):
    lc = (state.get("run_config") or {}).get("llm") or {}
    return (lc.get("nodes") or {}).get(node) or lc.get("default") or "gemini"


def _media_model_for(state, node):
    lc = (state.get("run_config") or {}).get("llm") or {}
    return (lc.get("nodes") or {}).get(node) or node   # "tts"/"stt" double as aliases


# `llm` namespace: node bodies below call llm.structured_call(...) etc. verbatim.
llm = SimpleNamespace(
    model_for=_model_for,
    media_model_for=_media_model_for,
    structured_call=lambda state, node, prompt, schema, *, system=None, images=None, **ov:
        complete_structured(_model_for(state, node), prompt, schema,
                            system=system, images=images, **ov),
    text_call=lambda state, node, prompt, *, system=None, images=None, **ov:
        complete_text(_model_for(state, node), prompt, system=system, images=images, **ov),
    speech_call=lambda state, node, text, *, voice="coral", instructions=None, **ov:
        complete_speech(_media_model_for(state, node), text, voice=voice,
                        instructions=instructions, **ov),
    transcribe_call=lambda state, node, file_path, *, language=None, **ov:
        complete_transcribe(_media_model_for(state, node), file_path, language=language, **ov),
)
print("llm layer ready:", list(MODEL_REGISTRY))


## 3 · Data models  *(viz_type + smiles restored)*

In [ ]:
# === Data models (inlined solvi/models.py) ===
# NOTE: `viz_type` and `smiles` are RESTORED on Classification here. The live
# repo dropped them, but route_engine / generate_rdkit / the organic verifier
# still read them — so this standalone copy re-adds both to make the whole graph
# run. Guidance for the classifier to emit them is restored in the classify cell.
from enum import Enum
from typing import Literal, Optional
from pydantic import BaseModel, Field, field_validator


class Subject(str, Enum):
    MATHS = "maths"
    PHYSICS = "physics"
    CHEMISTRY_ORGANIC = "chemistry_organic"
    CHEMISTRY_INORGANIC = "chemistry_inorganic"
    CHEMISTRY_PHYSICAL = "chemistry_physical"
    OTHER = "other"


class VizType(str, Enum):
    EQUATION_MORPH = "equation_morph"
    GRAPH_GEOMETRY = "graph_geometry"
    RAY_FIELD = "ray_field"
    SIMULATED_MOTION = "simulated_motion"
    STATIC_DIAGRAM = "static_diagram"
    MOLECULE_2D = "molecule_2d"
    MOLECULE_3D = "molecule_3d"
    NONE = "none"


class Engine(str, Enum):
    MANIM = "manim"
    RDKIT = "rdkit_js"
    MOL3D = "mol3d_js"
    MATTER = "matter_js"
    CANVAS = "canvas"
    TEXT = "text"


class ImageRef(BaseModel):
    kind: Literal["url", "base64"]
    data: str = Field(min_length=1)
    mime_type: str = "image/png"


class Option(BaseModel):
    text: str = ""
    images: list[ImageRef] = []


class Question(BaseModel):
    id: str
    text: str
    images: list[ImageRef] = []
    options: list[Option] = []
    answer_key: str                      # REQUIRED. No key -> unsupported.
    official_solution: Optional[str] = None
    subject_hint: Optional[Subject] = None
    source: Optional[str] = None

    @field_validator("options", mode="before")
    @classmethod
    def _coerce_plain_strings(cls, v):
        if isinstance(v, list):
            return [{"text": o} if isinstance(o, str) else o for o in v]
        return v

    def all_images(self) -> list[ImageRef]:
        imgs = list(self.images)
        for opt in self.options:
            imgs.extend(opt.images)
        return imgs

    def options_text(self) -> str:
        lines = []
        for i, opt in enumerate(self.options, start=1):
            suffix = " [see attached image]" if opt.images else ""
            lines.append(f"({i}) {opt.text}{suffix}")
        return "\n".join(lines)


class Classification(BaseModel):
    subject: Subject
    topic: str
    viz_type: VizType                    # RESTORED: drives engine routing
    smiles: Optional[str] = None         # RESTORED: consumed by rdkit/organic
    question_complexity: Literal["low", "medium", "high"]
    reasoning: str
    subject_confidence: float = Field(ge=0, le=1)
    complexity_confidence: float = Field(ge=0, le=1)
    # Syllabus knowledge-component tagging (filled by the kc_tag node)
    chapter_id: Optional[str] = None
    kc_ids: list[str] = []
    kc_confidence: Optional[float] = Field(default=None, ge=0, le=1)
    topics_complexity: Optional[Literal["low", "medium", "high"]] = None


class Step(BaseModel):
    title: str
    body: str
    why: str


class SolutionPlan(BaseModel):
    concept_primer: str
    approach: str
    steps: list[Step]
    answer: str
    trap: str
    fast_trick: Optional[str] = None
    narration: list[str]


class NarrationBeat(BaseModel):
    index: int
    text: str


class NarrationScript(BaseModel):
    language: Literal["hinglish", "english", "hindi"]
    beats: list[NarrationBeat]


class AudioSegment(BaseModel):
    index: int
    path: str
    duration: float
    transcript: Optional[str] = None
    match: Optional[float] = None


class AudioTrack(BaseModel):
    voice: str
    segments: list[AudioSegment]
    total: float


class Verification(BaseModel):
    ran: bool
    method: str
    agrees_with_key: Optional[bool] = None
    detail: str


class Solution(BaseModel):
    question_id: str
    status: Literal["ok", "requires_review", "unsupported"]
    artifact: Optional[str] = None
    artifact_type: Optional[Literal["html", "video"]] = None
    engine_used: Optional[Engine] = None
    plan: Optional[SolutionPlan] = None
    classification: Optional[Classification] = None
    narration_script: Optional[str] = None
    verification: Optional[Verification] = None
    confidence: float = 0.0
    flags: list[str] = []
    trace: list[str] = []
    reason: Optional[str] = None


print("models ready (viz_type + smiles restored on Classification)")


## 4 · Run config + graph state + config knobs

In [ ]:
# === Run config + graph state (inlined solvi/config.py + graph/state.py) ===
import operator
from typing import Annotated
from typing_extensions import TypedDict


class LLMConfig(BaseModel):
    default: str = "gemini"
    nodes: dict[str, str] = {}

    def for_node(self, node: str) -> str:
        return self.nodes.get(node, self.default)


class RunConfig(BaseModel):
    llm: LLMConfig = Field(default_factory=LLMConfig)
    min_confidence: float = 0.6
    on_low_confidence: Literal["text_only", "withhold"] = "text_only"
    routing_confidence_floor: float = 0.5
    disabled_engines: list[Engine] = []
    multipass_planning: bool = True
    narration: bool = True
    narration_language: Literal["hinglish", "english", "hindi"] = "hinglish"
    narration_voice: str = "coral"
    max_solve_attempts: int = 2
    max_render_attempts: int = 2
    max_audio_attempts: int = 2
    use_cache: bool = True


def _run_config(state) -> RunConfig:
    return RunConfig.model_validate(state.get("run_config") or {})


class SolviState(TypedDict, total=False):
    raw_input: dict
    run_config: dict
    question: Optional[Question]
    classification: Optional[Classification]
    plan: Optional[SolutionPlan]
    verification: Optional[Verification]
    engine: Optional[Engine]
    generated_code: Optional[str]
    layout_problems: list
    narration_script: Optional[NarrationScript]
    narration_problems: list
    audio_track: Optional[AudioTrack]
    sections_file: Optional[str]
    artifact: Optional[str]
    artifact_type: Optional[Literal["html", "video"]]
    confidence: float
    status: Literal["ok", "requires_review", "unsupported"]
    reason: Optional[str]
    flags: Annotated[list, operator.add]
    trace: Annotated[list, operator.add]
    solve_attempts: int
    render_attempts: int
    narration_attempts: int


# ------------------------------------------------------------------ config knobs
# Edit these, then re-run from here down. Gemini free tier 429s under load;
# nv-llama3.1-8b is the fast/reliable NIM fallback (needs NVIDIA_NIM_API_KEY).
MODELS = {
    "default": "gemini",
    # "classify": "nv-llama3.1-8b", "kc_tag": ..., "solve": ..., "verify": ...,
}
# narration=True exercises gen_script/gen_audio/audio_check/mux (needs OpenAI TTS
# + manim/ffmpeg). Leave False for a fast run; the audio nodes still exist and
# would degrade to a silent/text artifact if reached.
RUN = dict(narration=False, multipass_planning=True)

RUN_CONFIG = RunConfig(
    llm=LLMConfig(default=MODELS.get("default", "gemini"),
                  nodes={k: v for k, v in MODELS.items() if k != "default"}),
    **RUN,
).model_dump(mode="json")
print("run_config:", MODELS, "| narration:", RUN["narration"])


## 5 · Syllabus registry (loaded from file path)

In [ ]:
# === Syllabus registry (inlined solvi/syllabus.py) ===
# Loads the JEE syllabus JSON from SYLLABUS_PATH; used only by kc_tag.
from dataclasses import dataclass, field
from functools import lru_cache

SUBJECT_TO_CODE = {
    Subject.MATHS: "MATH",
    Subject.PHYSICS: "PHY",
    Subject.CHEMISTRY_ORGANIC: "CHEM",
    Subject.CHEMISTRY_INORGANIC: "CHEM",
    Subject.CHEMISTRY_PHYSICAL: "CHEM",
}


@dataclass(frozen=True)
class KnowledgeComponent:
    kc_id: str
    chapter_id: str
    sequence: int
    name: str


@dataclass(frozen=True)
class Chapter:
    chapter_id: str
    subject_code: str
    unit_number: int
    name: str
    kcs: tuple = field(default_factory=tuple)


class SyllabusRegistry:
    def __init__(self, path):
        raw = json.loads(Path(path).read_text())
        self.title = raw.get("syllabus", "")
        self.chapters_by_id = {}
        self.kcs_by_id = {}
        self.chapters_by_subject = {}
        for subj in raw["subjects"]:
            code = subj["subject_code"]
            chapters = []
            for ch in subj["chapters"]:
                kcs = tuple(
                    KnowledgeComponent(kc_id=kc["kc_id"], chapter_id=kc["chapter_id"],
                                       sequence=kc["sequence"], name=kc["knowledge_component"])
                    for kc in ch["knowledge_components"]
                )
                chapter = Chapter(chapter_id=ch["chapter_id"], subject_code=code,
                                  unit_number=ch["unit_number"], name=ch["chapter_name"], kcs=kcs)
                chapters.append(chapter)
                self.chapters_by_id[chapter.chapter_id] = chapter
                for kc in kcs:
                    self.kcs_by_id[kc.kc_id] = kc
            self.chapters_by_subject[code] = chapters

    def is_chapter(self, chapter_id):
        return chapter_id in self.chapters_by_id

    def validate_kcs(self, kc_ids):
        valid, invalid, seen = [], [], set()
        for kc_id in kc_ids:
            if kc_id in seen:
                continue
            seen.add(kc_id)
            (valid if kc_id in self.kcs_by_id else invalid).append(kc_id)
        return valid, invalid

    def candidate_block(self, subject):
        code = SUBJECT_TO_CODE.get(subject)
        if code is None:
            return ""
        lines = []
        for ch in self.chapters_by_subject.get(code, []):
            lines.append(f"{ch.chapter_id} — {ch.name}")
            for kc in ch.kcs:
                lines.append(f"  {kc.kc_id}: {kc.name}")
        return "\n".join(lines)


@lru_cache(maxsize=1)
def registry():
    return SyllabusRegistry(settings().syllabus_path)


_reg = registry()
print(f"syllabus: {len(_reg.chapters_by_id)} chapters, {len(_reg.kcs_by_id)} KCs")


## 6 · Node: ingest

In [ ]:
# === ingest node (inlined solvi/ingest/ingestor.py) ===
def _validate_llm_config(raw_llm):
    values = [raw_llm.get("default")] if raw_llm.get("default") else []
    values += list((raw_llm.get("nodes") or {}).values())
    for v in values:
        if not isinstance(v, str) or (v not in MODEL_REGISTRY and "/" not in v):
            return f"unknown model {v!r}: not a gateway alias and not a 'provider/model' string"
    return None


def _image_payload_bytes(q):
    total = 0
    for img in q.all_images():
        if img.kind == "base64":
            total += (len(img.data) * 3) // 4
    return total


def ingest(state):
    raw = state.get("raw_input") or {}
    if not raw.get("answer_key"):
        return {"status": "unsupported", "reason": "answer_key is required in v1",
                "trace": ["ingest: rejected — no answer key"]}
    llm_err = _validate_llm_config((state.get("run_config") or {}).get("llm") or {})
    if llm_err:
        return {"status": "unsupported", "reason": f"invalid llm config: {llm_err}",
                "trace": ["ingest: rejected — bad llm config"]}
    try:
        q = Question(**{k: v for k, v in raw.items() if not k.startswith("_")})
    except ValidationError as e:
        return {"status": "unsupported",
                "reason": f"invalid question payload: {e.errors()[0].get('msg', 'validation error')}",
                "trace": ["ingest: rejected — validation error"]}
    max_bytes = settings().max_image_bytes
    payload = _image_payload_bytes(q)
    if payload > max_bytes:
        return {"status": "unsupported",
                "reason": f"image payload {payload} bytes exceeds limit {max_bytes}",
                "trace": ["ingest: rejected — image payload too large"]}
    return {"question": q, "trace": [f"ingest: ok ({q.id})"]}


## 7 · Node: classify

In [ ]:
# === classify node (inlined solvi/classify/{rules,classifier}.py) ===
# rule_hints biases the LLM; ClassificationDraft is what the LLM returns.
# viz_type + smiles guidance RESTORED (from docs/IMPLEMENTATION_PLAN.md) so the
# classifier emits the fields the router/generators need.
import re as _re

_RULES = [
    (_re.compile(r"stereoisomer|enantiomer|diastereomer|IUPAC|SN1|SN2|Fehling|Tollens|"
                 r"aldol|Grignard|carbocation|electrophil|nucleophil|benzene|alkene|alkane|"
                 r"alkyne|functional group|chiral", _re.I),
     "likely chemistry_organic (molecule_2d)"),
    (_re.compile(r"\bcomplex\b.*\bion\b|ligand|CFSE|crystal field|coordination (number|compound)|"
                 r"oxidation state|d-orbital|octahedral|tetrahedral|square planar", _re.I),
     "likely chemistry_inorganic (molecule_3d)"),
    (_re.compile(r"lens|mirror|refractive index|refraction|fringe|diffraction|interference|"
                 r"prism|focal length|magnif|electric field|magnetic field|field lines", _re.I),
     "likely physics (ray_field)"),
    (_re.compile(r"projectile|collision|circular motion|pendulum|trajectory|inclined plane|"
                 r"free fall|spring.*oscillat", _re.I),
     "likely physics (simulated_motion)"),
    (_re.compile(r"circuit|resistor|capacitor|energy level|emf|Kirchhoff", _re.I),
     "likely physics (static_diagram)"),
    (_re.compile(r"assertion.*reason|statement I.*statement II|which of the following statements",
                 _re.I | _re.S),
     "statement/assertion-based — a visual may add nothing (viz none)"),
    (_re.compile(r"integral|derivative|area (of|enclosed|bounded)|parabola|ellipse|"
                 r"hyperbola|matrix|determinant|quadratic|locus", _re.I),
     "likely maths"),
]


def rule_hints(text):
    return [hint for pattern, hint in _RULES if pattern.search(text)]


class ClassificationDraft(BaseModel):
    subject: Subject
    topic: str
    viz_type: VizType
    smiles: Optional[str] = None
    question_complexity: str = Field(pattern="^(low|medium|high)$")
    reasoning: str
    subject_confidence: float = Field(ge=0, le=1)
    complexity_confidence: float = Field(ge=0, le=1)


CLASSIFY_PROMPT = """You classify exam questions by subject, difficulty, and the visual that best teaches them.

Question: {text}
Options:
{options}
Known answer: {answer_key}
Rule-based hints (may be wrong): {hints}

Field guidance:
- subject: one of [maths, physics, chemistry_organic, chemistry_inorganic, chemistry_physical, other]
- topic: the specific topic, e.g. 'definite integrals', 'crystal field theory'
- viz_type: pick what TEACHES best, not what looks impressive — one of:
    equation_morph  : the insight is algebraic manipulation (substitution, factoring, Vieta)
    graph_geometry  : the insight is spatial (areas, conics, intersections, loci)
    ray_field       : light paths or field lines must be traced
    simulated_motion: the student must SEE motion to believe it (projectile, circular)
    static_diagram  : a labelled diagram suffices (circuits, energy levels)
    molecule_2d     : 2D structure / mechanism / stereochemistry
    molecule_3d     : 3D geometry matters (octahedral, tetrahedral, orbital shapes)
    none            : purely conceptual / statement-based — a visual would add nothing
  If a visual would NOT genuinely aid understanding, return "none". Be honest.
- smiles: the SMILES string if a specific molecule is central to the question, else null
- question_complexity: low | medium | high — how hard this question is to solve
- reasoning: one sentence justifying the subject and the complexity call
- subject_confidence: 0.0-1.0 — your confidence in the subject classification
- complexity_confidence: 0.0-1.0 — your confidence in the complexity classification

Be honest about complexity: a routine one-step question is 'low'; multi-concept
or long derivations are 'high'."""


def classify_question(state):
    q = state["question"]
    hints = rule_hints(q.text) or ["none"]
    prompt = CLASSIFY_PROMPT.format(
        text=q.text, options=q.options_text() or "(no options)",
        answer_key=q.answer_key, hints="; ".join(hints))
    images = [img.model_dump() for img in q.all_images()]
    draft = llm.structured_call(state, "classify", prompt, ClassificationDraft,
                                images=images or None)
    return Classification(**draft.model_dump())


def classify(state):
    cls = classify_question(state)
    flags = []
    if cls.subject_confidence < 0.5:
        flags.append("low_classification_confidence")
    model = llm.model_for(state, "classify")
    return {"classification": cls, "flags": flags,
            "trace": [f"classify[{model}]: {cls.subject.value}/{cls.topic} "
                      f"(viz={cls.viz_type.value}, {cls.question_complexity}, "
                      f"subj_conf {cls.subject_confidence:.2f})"]}


## 8 · Node: kc_tag

In [ ]:
# === kc_tag node (inlined solvi/classify/kc_tagger.py) ===
class KCTag(BaseModel):
    chapter_id: str
    kc_ids: list[str] = Field(min_length=1, max_length=4)
    kc_confidence: float = Field(ge=0, le=1)
    topics_complexity: Literal["low", "medium", "high"]
    reasoning: str


KC_TAG_PROMPT = """Map this exam question to the official syllabus.

Question: {text}
Options:
{options}
Topic (from classification): {topic}

Below is the complete list of syllabus chapters and atomic knowledge
components (KCs) for this subject. Pick:
- chapter_id: the ONE chapter this question belongs to
- kc_ids: 1-4 KC ids the question actually tests (most central first).
  Use ONLY ids that appear in the list below.
- kc_confidence: 0.0-1.0
- topics_complexity: low | medium | high — the overall difficulty implied by
  the set of KCs this question tests (more KCs, or more advanced KCs, is harder)
- reasoning: one sentence

SYLLABUS:
{syllabus}"""


def tag_question(state):
    q = state["question"]
    cls = state["classification"]
    flags = []
    if cls.subject is Subject.OTHER:
        return cls, ["kc_tagging_skipped"]
    reg = registry()
    prompt = KC_TAG_PROMPT.format(
        text=q.text, options=q.options_text() or "(no options)",
        topic=cls.topic, syllabus=reg.candidate_block(cls.subject))
    tag = llm.structured_call(state, "kc_tag", prompt, KCTag)
    chapter_id = tag.chapter_id if reg.is_chapter(tag.chapter_id) else None
    if chapter_id is None:
        flags.append("invalid_kc_ids")
    valid, invalid = reg.validate_kcs(tag.kc_ids)
    if invalid and "invalid_kc_ids" not in flags:
        flags.append("invalid_kc_ids")
    updated = cls.model_copy(update={
        "chapter_id": chapter_id, "kc_ids": valid,
        "kc_confidence": tag.kc_confidence, "topics_complexity": tag.topics_complexity})
    return updated, flags


def kc_tag(state):
    try:
        updated, flags = tag_question(state)
    except Exception as e:
        return {"flags": ["kc_tagging_failed"],
                "trace": [f"kc_tag: failed — {type(e).__name__}: {e}"]}
    model = llm.model_for(state, "kc_tag")
    kc_desc = f"{updated.chapter_id} {updated.kc_ids}" if updated.chapter_id else "skipped"
    return {"classification": updated, "flags": flags, "trace": [f"kc_tag[{model}]: {kc_desc}"]}


## 9 · Node: solve (draft → critique → refine)

In [ ]:
# === solve node (inlined solvi/solve/{prompts,planner}.py) ===
# Multi-pass: draft -> self-critique -> refine (one round). On a verification
# retry the prior failure detail is injected as a critique into the draft prompt.
PLAN_SYSTEM = """You are a brilliant teacher writing a solution that TEACHES, not one that
just reaches the answer. You write concept-first lessons the way 3Blue1Brown
would: intuition before formula, reasoning beside every step."""

PLAN_PROMPT = """Question: {text}
Options:
{options}
CORRECT ANSWER (given — your derivation must reach this): {answer_key}
Official solution (reference, may be terse): {official_solution}
Classification: {subject} / {topic}
{critique_block}
Write a concept-first lesson with these fields:

- concept_primer: Teach the prerequisite idea FROM SCRATCH. No formula yet.
  Build intuition first. 3-5 sentences.
- approach: How does an expert DECIDE the method before computing? This is
  the part students miss most.
- steps: a list of steps, each with:
    - title: short label
    - body: the derivation line (use LaTeX with \\( \\) delimiters for maths)
    - why: the reasoning — REQUIRED, never leave empty
- answer: state it clearly
- trap: the SPECIFIC mistake students make on THIS question — not generic advice
- fast_trick: an exam shortcut, or null
- narration: one spoken line per beat, conversational, for voiceover

Rules:
- The concept primer must come BEFORE any formula. Intuition, then mathematics.
- Every step needs a 'why'. A step with no reasoning is a failed step.
- The trap must be specific. "Be careful with signs" is worthless. "Forgetting
  the factor of 2 only counts the area above the axis, giving 6pi-8" is useful.
- Your derivation MUST arrive at the given correct answer. If it doesn't, you
  have made an error — recheck."""

CRITIQUE_SYSTEM = """You are a ruthless reviewer of maths/science lessons. You check pedagogy,
not politeness."""

CRITIQUE_PROMPT = """Review this lesson plan for the question below. Answer these checks:
1. Does the concept primer teach from scratch, BEFORE any formula appears?
2. Does each step have a genuine 'why' (reasoning, not restatement)?
3. Is the trap specific to THIS question, not generic advice?
4. Does the derivation actually reach the given correct answer: {answer_key}?

Question: {text}

Lesson plan (JSON):
{plan_json}

Reply with:
- verdict: "pass" if all four checks pass, else "revise"
- critique: concrete, actionable problems (empty string if pass)"""

REFINE_PROMPT = """Your lesson plan was reviewed. Fix ALL the problems listed, keeping
everything that already works. Return the COMPLETE corrected lesson.

Question: {text}
Options:
{options}
CORRECT ANSWER (your derivation must reach this): {answer_key}

Previous plan (JSON):
{plan_json}

Reviewer's critique:
{critique}"""


class PlanCritique(BaseModel):
    verdict: Literal["pass", "revise"]
    critique: str = ""


def _base_prompt(state, critique):
    q = state["question"]
    cls = state["classification"]
    critique_block = ""
    if critique:
        critique_block = (
            "\nIMPORTANT — your previous derivation FAILED independent verification:\n"
            f"{critique}\n"
            "Rework the derivation; it must genuinely reach the given answer.\n")
    prompt = PLAN_PROMPT.format(
        text=q.text, options=q.options_text() or "(no options)",
        answer_key=q.answer_key, official_solution=q.official_solution or "(not provided)",
        subject=cls.subject.value, topic=cls.topic, critique_block=critique_block)
    images = [img.model_dump() for img in q.all_images()]
    return prompt, images


def _multipass_enabled(state):
    return bool((state.get("run_config") or {}).get("multipass_planning", True))


def _critique_and_refine(state, draft, images):
    q = state["question"]
    plan_json = draft.model_dump_json(indent=2)
    try:
        review = llm.structured_call(
            state, "solve",
            CRITIQUE_PROMPT.format(answer_key=q.answer_key, text=q.text, plan_json=plan_json),
            PlanCritique, system=CRITIQUE_SYSTEM)
        if review.verdict != "revise":
            return draft
        return llm.structured_call(
            state, "solve",
            REFINE_PROMPT.format(text=q.text, options=q.options_text() or "(no options)",
                                 answer_key=q.answer_key, plan_json=plan_json,
                                 critique=review.critique),
            SolutionPlan, system=PLAN_SYSTEM, images=images or None)
    except Exception:
        return draft


def plan_solution(state, critique=None):
    prompt, images = _base_prompt(state, critique)
    draft = llm.structured_call(state, "solve", prompt, SolutionPlan,
                                system=PLAN_SYSTEM, images=images or None)
    if not _multipass_enabled(state):
        return draft
    return _critique_and_refine(state, draft, images)


def solve(state):
    attempts = state.get("solve_attempts", 0) + 1
    prior = state.get("verification")
    critique = prior.detail if (prior and prior.agrees_with_key is False) else None
    plan = plan_solution(state, critique=critique)
    model = llm.model_for(state, "solve")
    return {"plan": plan, "solve_attempts": attempts,
            "trace": [f"solve[{model}]: plan generated (attempt {attempts})"]}


## 10 · Node: verify (compare ladder + sandboxed SymPy + rules)

In [ ]:
# === verify node (inlined solvi/verify/*) ===
# compare_answers ladder + a sandboxed subprocess that runs LLM-emitted SymPy,
# dispatched by subject. Never raises; degrades with flags.
import ast
import math
import subprocess
import tempfile
from typing import Sequence

# ---- compare.py: the answer-agreement ladder ----
REL_TOL = 1e-4
_COMPOUND_UNITS = (
    r"m/s\^2", r"m/s2", r"m/s²", r"m/s", r"km/h", r"cm", r"mm", r"km",
    r"ev", r"ohms", r"ohm", r"delta_o", r"delta o", r"δ₀", r"δo", r"deg(rees)?",
)
_UNIT_RE = _re.compile(
    r"(?:\s+|(?<=[\d\)]))\s*(?:" + "|".join(_COMPOUND_UNITS) + r"|[amsjnvw])\s*$", _re.IGNORECASE)
_OPTION_INDEX_RE = _re.compile(r"^\s*[\(\[]?\s*([1-9]\d*|[A-Da-d])\s*[\)\]\.]?\s*$")


def normalize(s):
    s = s.strip().replace("π", "pi").replace("≈", "").replace("−", "-")
    s = s.replace("Δ₀", "delta_o").replace("Δo", "delta_o").replace("Δ", "delta")
    s = _re.sub(r"\s+", " ", s).lower()
    prev = None
    while prev != s:
        prev = s
        s = _UNIT_RE.sub("", s).strip()
    return s


def option_index(answer_key, n_options):
    m = _OPTION_INDEX_RE.match(answer_key or "")
    if not m:
        return None
    token = m.group(1)
    idx = ord(token.upper()) - ord("A") + 1 if token.isalpha() else int(token)
    return idx if 1 <= idx <= n_options else None


def resolve_answer_key(answer_key, options):
    idx = option_index(answer_key, len(options))
    return options[idx - 1].text if idx is not None else answer_key


def _sympify_safe(s):
    from sympy.parsing.sympy_parser import (convert_xor, implicit_multiplication_application,
                                            parse_expr, standard_transformations)
    cleaned = s.replace("^", "**").strip()
    if not cleaned:
        return None
    try:
        return parse_expr(cleaned, transformations=standard_transformations
                          + (implicit_multiplication_application, convert_xor), evaluate=True)
    except Exception:
        return None


def _as_float(expr_or_str):
    try:
        if isinstance(expr_or_str, str):
            return float(expr_or_str)
        v = complex(expr_or_str.evalf())
        return None if abs(v.imag) > 1e-12 else float(v.real)
    except Exception:
        return None


def _compare_single(computed, target):
    a, b = normalize(computed), normalize(target)
    if a and a == b:
        return True, f"exact match ({a!r})"
    ea, eb = _sympify_safe(a), _sympify_safe(b)
    if ea is not None and eb is not None:
        try:
            if ea.equals(eb) is True:
                return True, f"symbolic match ({ea} == {eb})"
        except Exception:
            pass
        try:
            import sympy
            if sympy.simplify(ea - eb) == 0:
                return True, f"symbolic match via simplify ({ea} == {eb})"
        except Exception:
            pass
    fa = _as_float(ea) if ea is not None else _as_float(a)
    fb = _as_float(eb) if eb is not None else _as_float(b)
    if fa is not None and fb is not None:
        denom = max(abs(fa), abs(fb), 1e-12)
        if abs(fa - fb) / denom <= REL_TOL:
            return True, f"numeric match ({fa} ≈ {fb}, rel tol {REL_TOL})"
        return False, f"numeric mismatch ({fa} vs {fb})"
    return False, f"no match ({a!r} vs {b!r})"


def compare_answers(computed, answer_key, options=None):
    options = list(options or [])
    targets = []
    idx = option_index(answer_key, len(options))
    if idx is not None:
        targets.append((f"option ({idx}) text", options[idx - 1].text))
        targets.append(("raw option index", answer_key))
    else:
        targets.append(("answer key", answer_key))
    failures = []
    for label, target in targets:
        ok, detail = _compare_single(computed, target)
        if ok:
            return True, f"{detail} against {label} {target!r}"
        failures.append(f"{detail} against {label} {target!r}")
    return False, "; ".join(failures)


# ---- safe_exec.py: AST whitelist + isolated subprocess ----
ALLOWED_IMPORTS = {"sympy", "math", "fractions", "itertools"}
BANNED_CALLS = {"open", "exec", "eval", "compile", "__import__", "input", "breakpoint",
                "getattr", "setattr", "delattr", "vars", "globals", "locals"}


def _is_dunder(name):
    return name.startswith("__") and name.endswith("__")


def check_code(code):
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"syntax error: {e.msg} (line {e.lineno})"
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_IMPORTS:
                    return f"import of {alias.name!r} is not allowed"
        elif isinstance(node, ast.ImportFrom):
            if node.level:
                return "relative imports are not allowed"
            if (node.module or "").split(".")[0] not in ALLOWED_IMPORTS:
                return f"import from {node.module!r} is not allowed"
        elif isinstance(node, ast.Attribute):
            if _is_dunder(node.attr):
                return f"dunder attribute access ({node.attr!r}) is not allowed"
        elif isinstance(node, ast.Name):
            if _is_dunder(node.id):
                return f"dunder name ({node.id!r}) is not allowed"
        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in BANNED_CALLS:
                return f"call to {node.func.id!r} is not allowed"
    return None


_HARNESS = """\
import contextlib as _contextlib
import io as _io
import json as _json
import sys as _sys

_SRC = {src!r}
_ns = {{}}
_buf = _io.StringIO()
try:
    with _contextlib.redirect_stdout(_buf):
        exec(compile(_SRC, "<llm_code>", "exec"), _ns)
except Exception as _e:
    print(_json.dumps({{"error": f"{{type(_e).__name__}}: {{_e}}"}}))
    _sys.exit(3)
if "result" not in _ns:
    print(_json.dumps({{"error": "code did not define a variable named 'result'"}}))
    _sys.exit(3)
try:
    print(_json.dumps({{"result": repr(_ns["result"])}}))
except Exception as _e:
    print(_json.dumps({{"error": f"result not representable: {{_e}}"}}))
    _sys.exit(3)
"""


def _posix_limits(timeout):
    def apply():
        import resource
        cpu = max(1, int(timeout))
        resource.setrlimit(resource.RLIMIT_CPU, (cpu, cpu + 2))
    return apply


def safe_exec(code, timeout=None):
    if timeout is None:
        timeout = settings().safe_exec_timeout
    err = check_code(code)
    if err:
        return {"ok": False, "error": f"code rejected: {err}"}
    path = None
    try:
        fd, path = tempfile.mkstemp(suffix=".py", prefix="solvi_safe_exec_")
        with os.fdopen(fd, "w") as fh:
            fh.write(_HARNESS.format(src=code))
        kwargs = {}
        if os.name == "posix":
            kwargs["preexec_fn"] = _posix_limits(timeout)
        try:
            proc = subprocess.run([sys.executable, "-I", path], capture_output=True,
                                  text=True, timeout=timeout, **kwargs)
        except subprocess.TimeoutExpired:
            return {"ok": False, "error": f"timed out after {timeout}s"}
        lines = [ln for ln in proc.stdout.strip().splitlines() if ln.strip()]
        if not lines:
            tail = (proc.stderr or "").strip()[-300:]
            return {"ok": False, "error": f"no output (exit {proc.returncode}): {tail}"}
        try:
            payload = json.loads(lines[-1])
        except json.JSONDecodeError:
            return {"ok": False, "error": f"unparseable output: {lines[-1][:200]}"}
        if "result" in payload:
            return {"ok": True, "result": str(payload["result"])}
        return {"ok": False, "error": str(payload.get("error", "unknown error"))}
    except Exception as e:
        return {"ok": False, "error": f"safe_exec internal error: {type(e).__name__}: {e}"}
    finally:
        if path:
            try:
                os.unlink(path)
            except OSError:
                pass


# ---- math_verifier.py + physics_verifier.py (LLM-emitted SymPy) ----
class SympyProgram(BaseModel):
    code: str
    expected_form: str


VERIFY_SYSTEM = """You write short, standalone SymPy programs that independently \
verify exam answers. You recompute from first principles — you never transcribe \
someone else's derivation."""

MATH_VERIFY_PROMPT = """Write a STANDALONE Python/SymPy program that INDEPENDENTLY \
recomputes the final answer to this question from the question itself.

Question: {text}
Options:
{options}
Answer key (for context only — do NOT hardcode it): {answer_key}
The solution plan claims the answer is: {plan_answer}
(Do NOT transcribe the plan's steps. Recompute independently from the question.)

Program requirements (strict):
- `import sympy` explicitly. No other imports except math, fractions, itertools.
- Define a variable named `result` holding the final answer (a sympy object
  or a number). If the answer is one of the options, `result` should be the
  VALUE, not the option number.
- No print statements. No file/network access. No input().
- Keep it short and exact — prefer symbolic computation over floats.

Also state `expected_form`: one phrase describing the expected shape of the
result (e.g. "a rational number", "an expression in pi")."""

PHYSICS_VERIFY_PROMPT = """Write a STANDALONE Python/SymPy program that INDEPENDENTLY \
recomputes the final answer to this physics question from the question itself.

Question: {text}
Options:
{options}
Answer key (for context only — do NOT hardcode it): {answer_key}
The solution plan claims the answer is: {plan_answer}
(Do NOT transcribe the plan's steps. Recompute independently from the question.)

Program requirements (strict):
- `import sympy` explicitly. No other imports except math, fractions, itertools.
- Work in SI units with the numeric values given in the question (use the
  stated constants, e.g. g = 10 m/s^2 if the question says so).
- CHECK DIMENSIONAL CONSISTENCY of the governing equation as you set it up:
  a formula whose sides have different dimensions is wrong — rethink it.
- Define a variable named `result` holding the final NUMERIC or symbolic
  answer (magnitude in the units the question asks for; if the answer is one
  of the options, `result` is the VALUE, not the option number).
- No print statements. No file/network access. No input().

Also state `expected_form`: one phrase describing the expected shape of the
result (e.g. "a speed in m/s", "an energy in eV")."""


def _emit_and_run(state, prompt):
    q = state["question"]
    try:
        prog = llm.structured_call(state, "verify", prompt, SympyProgram, system=VERIFY_SYSTEM)
    except Exception as e:
        return Verification(ran=False, method="sympy", agrees_with_key=None,
                            detail=f"code emission failed: {type(e).__name__}: {e}")
    err = check_code(prog.code)
    if err:
        return Verification(ran=False, method="sympy", agrees_with_key=None,
                            detail=f"emitted code rejected: {err}")
    res = safe_exec(prog.code)
    if not res.get("ok"):
        return Verification(ran=False, method="sympy", agrees_with_key=None,
                            detail=f"exec failed: {res.get('error', 'unknown error')}")
    try:
        agrees, detail = compare_answers(res["result"], q.answer_key, q.options)
    except Exception as e:
        return Verification(ran=False, method="sympy", agrees_with_key=None,
                            detail=f"comparison failed: {type(e).__name__}: {e}")
    return Verification(ran=True, method="sympy", agrees_with_key=agrees,
                        detail=f"recomputed {res['result']!r} (expected {prog.expected_form}); {detail}")


def verify_math(state):
    try:
        q, plan = state["question"], state.get("plan")
        prompt = MATH_VERIFY_PROMPT.format(text=q.text, options=q.options_text() or "(no options)",
                                           answer_key=q.answer_key,
                                           plan_answer=plan.answer if plan else "(no plan)")
        return _emit_and_run(state, prompt)
    except Exception as e:
        return Verification(ran=False, method="sympy", agrees_with_key=None,
                            detail=f"verifier error: {type(e).__name__}: {e}")


def verify_physics(state):
    try:
        q, plan = state["question"], state.get("plan")
        prompt = PHYSICS_VERIFY_PROMPT.format(text=q.text, options=q.options_text() or "(no options)",
                                              answer_key=q.answer_key,
                                              plan_answer=plan.answer if plan else "(no plan)")
        return _emit_and_run(state, prompt)
    except Exception as e:
        return Verification(ran=False, method="sympy", agrees_with_key=None,
                            detail=f"verifier error: {type(e).__name__}: {e}")


# ---- inorganic_verifier.py (deterministic rules, no LLM) ----
_CFSE_RE = _re.compile(r"CFSE|crystal field stabili[sz]ation", _re.IGNORECASE)
_GEOMETRY_RE = _re.compile(r"geometry of", _re.IGNORECASE)
_D_COUNT_RE = _re.compile(r"\bd[\^ ]?(\d+)\b", _re.IGNORECASE)
_LOW_SPIN_RE = _re.compile(r"low[\s-]*spin", _re.IGNORECASE)
_HIGH_SPIN_RE = _re.compile(r"high[\s-]*spin", _re.IGNORECASE)
_COMPLEX_RE = _re.compile(r"\[([^\]]+)\]")
_LIGAND_RE = _re.compile(r"\((\w+?)\)(\d*)")
_D8_METAL_RE = _re.compile(r"\b(Ni|Pd|Pt)\b")
_STRONG_FIELD_RE = _re.compile(r"strong[\s-]*field", _re.IGNORECASE)
GEOMETRY_BY_CN = {2: "linear", 6: "octahedral"}


def octahedral_filling(n_d, spin):
    if spin == "low":
        t2g = min(n_d, 6)
        return t2g, n_d - t2g
    first = min(n_d, 5)
    second = n_d - first
    t2g = min(first, 3) + min(second, 3)
    eg = (first - min(first, 3)) + (second - min(second, 3))
    return t2g, eg


def _verify_cfse(q):
    if _re.search(r"tetrahedral", q.text, _re.IGNORECASE):
        return Verification(ran=False, method="rules", agrees_with_key=None,
                            detail="tetrahedral CFSE not supported by the rule engine")
    m = _D_COUNT_RE.search(q.text)
    if not m:
        return Verification(ran=False, method="rules", agrees_with_key=None,
                            detail="could not parse d-electron count from question")
    n_d = int(m.group(1))
    if not 0 <= n_d <= 10:
        return Verification(ran=False, method="rules", agrees_with_key=None,
                            detail=f"implausible d-electron count d{n_d}")
    spin = None
    if _LOW_SPIN_RE.search(q.text):
        spin = "low"
    elif _HIGH_SPIN_RE.search(q.text):
        spin = "high"
    if spin is None:
        if 4 <= n_d <= 7:
            return Verification(ran=False, method="rules", agrees_with_key=None,
                                detail=f"d{n_d} is spin-state dependent and the question "
                                       "does not specify low/high spin")
        spin = "high"
    t2g, eg = octahedral_filling(n_d, spin)
    cfse = round(-0.4 * t2g + 0.6 * eg, 10)
    computed = f"{cfse:g}"
    agrees, detail = compare_answers(computed, q.answer_key, q.options)
    return Verification(ran=True, method="rules", agrees_with_key=agrees,
                        detail=f"d{n_d} {spin}-spin octahedral: t2g^{t2g} eg^{eg} → "
                               f"CFSE = {computed} Δo; {detail}")


def _coordination_number(text):
    m = _COMPLEX_RE.search(text)
    if not m:
        return None
    groups = _LIGAND_RE.findall(m.group(1))
    if not groups:
        return None
    return sum(int(count) if count else 1 for _, count in groups)


def _verify_geometry(q):
    cn = _coordination_number(q.text)
    if cn is None:
        return Verification(ran=False, method="rules", agrees_with_key=None,
                            detail="could not parse a coordination number from the question")
    resolved = resolve_answer_key(q.answer_key, q.options).strip().lower()
    if cn == 4:
        strong_field = bool(_STRONG_FIELD_RE.search(q.text)) or "(CN)" in q.text
        if _D8_METAL_RE.search(q.text) and strong_field:
            geometry = "square planar"
        else:
            return Verification(ran=True, method="rules", agrees_with_key=None,
                                detail="CN 4 → tetrahedral or square planar; cannot decide "
                                       f"without a d8 metal + strong-field ligand (key: {resolved!r})")
    elif cn in GEOMETRY_BY_CN:
        geometry = GEOMETRY_BY_CN[cn]
    else:
        return Verification(ran=False, method="rules", agrees_with_key=None,
                            detail=f"no geometry rule for coordination number {cn}")
    agrees = geometry in resolved
    return Verification(ran=True, method="rules", agrees_with_key=agrees,
                        detail=f"coordination number {cn} → {geometry}; answer key resolves "
                               f"to {resolved!r}")


def verify_inorganic(state):
    try:
        q = state["question"]
        if _CFSE_RE.search(q.text):
            return _verify_cfse(q)
        if _GEOMETRY_RE.search(q.text):
            return _verify_geometry(q)
        return Verification(ran=False, method="rules", agrees_with_key=None, detail="no rule matched")
    except Exception as e:
        return Verification(ran=False, method="rules", agrees_with_key=None,
                            detail=f"verifier error: {type(e).__name__}: {e}")


# ---- organic_verifier.py (RDKit; needs cls.smiles, now restored) ----
_STEREO_RE = _re.compile(r"how many stereoisomer|number of stereoisomer", _re.IGNORECASE)


def verify_organic(state):
    try:
        from rdkit import Chem, RDLogger
        from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers
        RDLogger.DisableLog("rdApp.*")
        q = state["question"]
        cls = state.get("classification")
        smiles = cls.smiles if cls else None
        if not smiles:
            return Verification(ran=False, method="none", agrees_with_key=None,
                                detail="no SMILES to verify")
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return Verification(ran=False, method="rdkit", agrees_with_key=None,
                                detail=f"invalid SMILES: {smiles!r}")
        if _STEREO_RE.search(q.text):
            n = len(list(EnumerateStereoisomers(mol)))
            agrees, detail = compare_answers(str(n), q.answer_key, q.options)
            return Verification(ran=True, method="rdkit", agrees_with_key=agrees,
                                detail=f"RDKit enumerated {n} stereoisomers for {smiles!r}; {detail}")
        return Verification(ran=False, method="rdkit", agrees_with_key=None,
                            detail=f"SMILES valid ({smiles!r}); no rule for this question type")
    except Exception as e:
        return Verification(ran=False, method="rdkit", agrees_with_key=None,
                            detail=f"verifier error: {type(e).__name__}: {e}")


# ---- base.py: dispatch by subject ----
def run_verification(state):
    try:
        cls = state.get("classification")
        subject = cls.subject if cls else None
        if subject is Subject.MATHS:
            return verify_math(state)
        if subject is Subject.PHYSICS:
            return verify_physics(state)
        if subject is Subject.CHEMISTRY_ORGANIC:
            return verify_organic(state)
        if subject is Subject.CHEMISTRY_INORGANIC:
            return verify_inorganic(state)
        if subject is Subject.CHEMISTRY_PHYSICAL:
            ver = verify_inorganic(state)
            return ver if ver.ran else verify_physics(state)
        return Verification(ran=False, method="none", agrees_with_key=None,
                            detail="unsupported subject")
    except Exception as e:
        return Verification(ran=False, method="none", agrees_with_key=None,
                            detail=f"verification error: {type(e).__name__}: {e}")


def verify(state):
    ver = run_verification(state)
    flags = []
    if ver.agrees_with_key is False:
        flags.append("VERIFICATION_CONTRADICTS_KEY")
    elif ver.agrees_with_key is None:
        flags.append("unverified")
    if ver.method == "rdkit" and "invalid SMILES" in ver.detail:
        flags.append("invalid_smiles")
    if ver.agrees_with_key is True and state.get("solve_attempts", 0) > 1:
        flags.append("retry_succeeded")
    detail = ver.detail if len(ver.detail) <= 80 else ver.detail[:80] + "…"
    return {"verification": ver, "flags": flags,
            "trace": [f"verify[{ver.method}]: agrees={ver.agrees_with_key} — {detail}"]}


## 11 · HTML shell (base template + render_page)

In [ ]:
# === HTML shell (inlined solvi/generate/{html.py, templates/base.html.j2}) ===
# One self-contained dark-theme page: concept-first stepper, MathJax, narration
# captions, visible CDN-failure banner. Engines contribute engine_head/viz_panel/
# viz_js. `r"""..."""` keeps the template's \\( \\) backslashes exactly.
from jinja2 import Environment, select_autoescape

BASE_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Solvi — {{ cls.topic }}</title>
<style>
  :root {
    --bg: #0d0e12; --panel: #14161d; --card: #1a1d26;
    --amber: #ffb454; --blue: #58b8ff; --teal: #3ad6c5;
    --red: #ff6b6b; --dim: #9aa0ad; --text: #e8eaf0;
    --border: #262a36;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: var(--bg); color: var(--text);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    line-height: 1.6; min-height: 100vh;
    display: flex; flex-direction: column; align-items: center;
    padding: 24px 16px 110px;
  }
  .page { width: 100%; max-width: 860px; }
  header.meta { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 16px; }
  .chip {
    font-size: 12px; padding: 3px 10px; border-radius: 999px;
    border: 1px solid var(--border); color: var(--dim); background: var(--panel);
  }
  .chip.subject { color: var(--amber); border-color: color-mix(in srgb, var(--amber) 40%, transparent); }
  .chip.kc { color: var(--teal); }
  .question-card {
    background: var(--panel); border: 1px solid var(--border);
    border-radius: 12px; padding: 18px 20px; margin-bottom: 20px;
  }
  .question-card .qlabel { color: var(--dim); font-size: 12px; letter-spacing: .08em;
    text-transform: uppercase; margin-bottom: 6px; }
  .question-card img { max-width: 100%; border-radius: 8px; margin-top: 10px; }
  .options { margin-top: 10px; color: var(--dim); }
  .options li { margin-left: 20px; }

  .lesson { background: var(--card); border: 1px solid var(--border);
    border-radius: 12px; overflow: hidden; }
  .stage { display: none; padding: 26px 28px; min-height: 220px; }
  .stage.active { display: block; }
  .stage h2 { font-size: 22px; margin-bottom: 12px; }
  .stage .body { font-size: 17px; }
  .stage .why {
    margin-top: 14px; padding: 10px 14px; border-left: 3px solid var(--teal);
    background: color-mix(in srgb, var(--teal) 7%, transparent);
    color: var(--dim); font-size: 15px; border-radius: 0 8px 8px 0;
  }
  .stage .why b { color: var(--teal); }
  .stage#stage-concept h2 { color: var(--blue); }
  .stage#stage-answer h2 { color: var(--amber); }
  .stage#stage-trap h2 { color: var(--red); }
  .stage#stage-trick h2 { color: var(--teal); }

  .viz-wrap { padding: 0 28px 22px; }
  #viz-panel { border: 1px solid var(--border); border-radius: 10px;
    background: var(--panel); overflow: hidden; }
  .cdn-fail { padding: 24px; color: var(--red); text-align: center; font-size: 15px; }

  nav.stepper {
    position: fixed; bottom: 0; left: 0; right: 0;
    background: color-mix(in srgb, var(--bg) 88%, transparent);
    backdrop-filter: blur(10px); border-top: 1px solid var(--border);
    display: flex; align-items: center; justify-content: center; gap: 18px;
    padding: 14px 16px;
  }
  nav.stepper button {
    background: var(--panel); color: var(--text); border: 1px solid var(--border);
    padding: 8px 20px; border-radius: 8px; font-size: 15px; cursor: pointer;
  }
  nav.stepper button:hover:not(:disabled) { border-color: var(--blue); color: var(--blue); }
  nav.stepper button:disabled { opacity: .35; cursor: default; }
  .dots { display: flex; gap: 7px; }
  .dots span { width: 9px; height: 9px; border-radius: 50%; background: var(--border);
    cursor: pointer; transition: background .2s; }
  .dots span.on { background: var(--blue); }
  .caption {
    margin-top: 18px; text-align: center; color: var(--dim); font-style: italic;
    font-size: 15px; min-height: 24px; padding: 0 20px;
  }
  .fallback-banner {
    background: color-mix(in srgb, var(--red) 12%, transparent);
    border: 1px solid var(--red); color: var(--red);
    padding: 8px 14px; border-radius: 8px; margin-bottom: 14px; font-size: 14px;
    display: none;
  }
</style>
<script>
  window.MathJax = { tex: { inlineMath: [["\\(", "\\)"], ["$", "$"]] }, svg: { fontCache: "global" } };
  function solviCdnFail(what) {
    var b = document.getElementById("fallback-banner");
    if (b) { b.style.display = "block"; b.textContent = "Could not load " + what + ". Check your connection."; }
  }
</script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/3.2.2/es5/tex-svg.min.js"
        onerror="solviCdnFail('MathJax (math rendering)')" async></script>
{{ engine_head | safe }}
</head>
<body>
<div class="page">
  <div id="fallback-banner" class="fallback-banner"></div>

  <header class="meta">
    <span class="chip subject">{{ cls.subject.value }}</span>
    <span class="chip">{{ cls.topic }}</span>
    <span class="chip">engine: {{ engine_label }}</span>
    {% if cls.chapter_id %}<span class="chip kc">{{ cls.chapter_id }}</span>{% endif %}
    {% for kc in cls.kc_ids %}<span class="chip kc">{{ kc }}</span>{% endfor %}
  </header>

  <section class="question-card">
    <div class="qlabel">Question {{ question.id }}</div>
    <div>{{ question.text }}</div>
    {% for src in question_images %}<img src="{{ src }}" alt="question figure">{% endfor %}
    {% if question.options %}
    <ol class="options">
      {% for opt in question.options %}<li>{{ opt.text }}</li>{% endfor %}
    </ol>
    {% endif %}
  </section>

  <main class="lesson">
    {% for stage in stages %}
    <section class="stage{% if loop.first %} active{% endif %}" id="stage-{{ stage.id }}">
      <h2>{{ stage.title }}</h2>
      <div class="body">{{ stage.body }}</div>
      {% if stage.why %}<div class="why"><b>Why:</b> {{ stage.why }}</div>{% endif %}
    </section>
    {% endfor %}
    {% if viz_panel %}
    <div class="viz-wrap"><div id="viz-panel">{{ viz_panel | safe }}</div></div>
    {% endif %}
  </main>

  <div class="caption" id="caption"></div>
</div>

<nav class="stepper">
  <button id="prev">&larr; Back</button>
  <div class="dots" id="dots"></div>
  <button id="next">Next &rarr;</button>
</nav>

<script>
  const STAGES = {{ stages | tojson }};
  let current = 0;

  const dots = document.getElementById("dots");
  STAGES.forEach((s, i) => {
    const d = document.createElement("span");
    d.title = s.label;
    d.addEventListener("click", () => go(i));
    dots.appendChild(d);
  });

  function go(i) {
    if (i < 0 || i >= STAGES.length) return;
    current = i;
    document.querySelectorAll(".stage").forEach((el, j) =>
      el.classList.toggle("active", j === i));
    document.querySelectorAll("#dots span").forEach((el, j) =>
      el.classList.toggle("on", j === i));
    document.getElementById("prev").disabled = i === 0;
    document.getElementById("next").disabled = i === STAGES.length - 1;
    document.getElementById("caption").textContent = STAGES[i].narration || "";
    if (window.SolviViz && typeof window.SolviViz.onStage === "function") {
      try { window.SolviViz.onStage(i, STAGES[i].id); } catch (e) { console.error(e); }
    }
  }

  document.getElementById("prev").addEventListener("click", () => go(current - 1));
  document.getElementById("next").addEventListener("click", () => go(current + 1));
  document.addEventListener("keydown", (e) => {
    if (e.key === "ArrowRight") go(current + 1);
    if (e.key === "ArrowLeft") go(current - 1);
  });

  window.addEventListener("load", () => {
    if (window.SolviViz && typeof window.SolviViz.init === "function") {
      try { window.SolviViz.init(); } catch (e) { console.error(e); }
    }
  });
  go(0);
</script>
{{ viz_js | safe }}
</body>
</html>"""

_JENV = Environment(autoescape=select_autoescape(["html", "j2"]))
_TEMPLATE = _JENV.from_string(BASE_TEMPLATE)


def image_src(img):
    return img.data if img.kind == "url" else f"data:{img.mime_type};base64,{img.data}"


def stages_from_plan(plan):
    stages = [
        {"id": "concept", "label": "Concept", "title": "The Core Idea",
         "body": plan.concept_primer, "why": ""},
        {"id": "approach", "label": "Approach", "title": "Choosing the Method",
         "body": plan.approach, "why": ""},
    ]
    for i, step in enumerate(plan.steps, start=1):
        stages.append({"id": f"step-{i}", "label": f"Step {i}",
                       "title": step.title, "body": step.body, "why": step.why})
    stages.append({"id": "answer", "label": "Answer", "title": "The Answer",
                   "body": plan.answer, "why": ""})
    stages.append({"id": "trap", "label": "Trap", "title": "The Trap",
                   "body": plan.trap, "why": ""})
    if plan.fast_trick:
        stages.append({"id": "trick", "label": "Trick", "title": "Exam Shortcut",
                       "body": plan.fast_trick, "why": ""})
    narration = list(plan.narration)
    for i, stage in enumerate(stages):
        stage["narration"] = narration[i] if i < len(narration) else ""
    return stages


def render_page(question, cls, plan, *, engine_head="", viz_panel="", viz_js="",
                engine_label="text"):
    return _TEMPLATE.render(
        question=question, cls=cls, stages=stages_from_plan(plan),
        question_images=[image_src(i) for i in question.images],
        engine_head=engine_head, viz_panel=viz_panel, viz_js=viz_js,
        engine_label=engine_label)


class GeneratedArtifact:
    def __init__(self, content, type):
        self.content, self.type = content, type


class TextGenerator:
    def generate(self, q, cls, plan):
        return GeneratedArtifact(render_page(q, cls, plan, engine_label="text"), "html")


## 12 · Generators: text, canvas, matter

In [ ]:
# === HTML generator nodes (inlined solvi/generate/{text,canvas,matter,mol3d,rdkit}_gen.py) ===
# Merged into one namespace, so per-engine globals are given unique names
# (_fb_*, *_CDN, *_JS) to avoid collisions. Behaviour is verbatim.
import math
from html import escape

_TEXT_GEN = TextGenerator()


def gen_text(state):
    art = _TEXT_GEN.generate(state["question"], state["classification"], state["plan"])
    return {"artifact": art.content, "artifact_type": art.type,
            "engine": Engine.TEXT, "trace": ["generate[text]: produced html"]}


# --------------------------------------------------------------- canvas (SVG) ---
CANVAS_W, CANVAS_H = 800, 450
KNOWN_TYPES = {"rect", "circle", "line", "arrow", "text", "resistor", "battery", "level"}


class CanvasElement(BaseModel):
    type: str
    x: float = 0.0
    y: float = 0.0
    x2: Optional[float] = None
    y2: Optional[float] = None
    width: Optional[float] = None
    height: Optional[float] = None
    radius: Optional[float] = None
    label: str = ""
    color: str = "#58b8ff"


class CanvasSpec(BaseModel):
    elements: list[CanvasElement]
    caption: str = ""


CANVAS_PROMPT = """Design a labelled static diagram for this lesson as a flat element list.

Question: {text}
Known answer: {answer_key}
Lesson plan (JSON):
{plan_json}

The canvas is {w} wide x {h} high; x grows right, y grows DOWN.
Element types (only these):
- rect:     x, y, width, height             (box / block / plate)
- circle:   x, y, radius                    (node / bob / lens)
- line:     x, y to x2, y2                  (wire / surface / axis)
- arrow:    x, y to x2, y2 (head at x2,y2)  (force / current / ray)
- text:     x, y, label                     (free-standing annotation)
- resistor: x, y to x2, y2                  (drawn as a zigzag)
- battery:  x, y to x2, y2                  (long/short plates; + at x2,y2)
- level:    x, y, width, label              (horizontal energy level)
Every element may carry a short label and a hex color.
Lay the diagram out clearly: no overlaps, generous spacing, margins of at
least 40 on every side. 6-20 elements is the right size."""

_COLOR_RE = _re.compile(r"^#[0-9a-fA-F]{3,8}$|^[a-zA-Z]+$")


def _color(c, default="#58b8ff"):
    return c if c and _COLOR_RE.match(c) else default


def _text_el(x, y, label, color, anchor="middle", size=14):
    return (f'<text x="{x:.1f}" y="{y:.1f}" fill="{color}" '
            f'font-size="{size}" text-anchor="{anchor}" '
            f'font-family="sans-serif">{escape(label)}</text>')


def _axis_units(x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    length = math.hypot(dx, dy) or 1.0
    ux, uy = dx / length, dy / length
    return ux, uy, -uy, ux, length


def _resistor(e, color):
    x1, y1 = e.x, e.y
    x2 = e.x2 if e.x2 is not None else e.x + 100
    y2 = e.y2 if e.y2 is not None else e.y
    ux, uy, px, py, length = _axis_units(x1, y1, x2, y2)
    lead = min(max(8.0, length * 0.2), length / 2)
    amp, n = 8.0, 6
    zig = length - 2 * lead
    pts = [(x1, y1), (x1 + ux * lead, y1 + uy * lead)]
    for k in range(1, n + 1):
        t = lead + zig * (k - 0.5) / n
        side = amp if k % 2 else -amp
        pts.append((x1 + ux * t + px * side, y1 + uy * t + py * side))
    pts.append((x2 - ux * lead, y2 - uy * lead))
    pts.append((x2, y2))
    poly = " ".join(f"{a:.1f},{b:.1f}" for a, b in pts)
    out = (f'<polyline points="{poly}" fill="none" stroke="{color}" '
           'stroke-width="2" stroke-linejoin="round"/>')
    if e.label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        out += _text_el(mx + px * (amp + 14), my + py * (amp + 14), e.label, "#e8eaf0")
    return out


def _battery(e, color):
    x1, y1 = e.x, e.y
    x2 = e.x2 if e.x2 is not None else e.x + 60
    y2 = e.y2 if e.y2 is not None else e.y
    ux, uy, px, py, _length = _axis_units(x1, y1, x2, y2)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    gap, long_p, short_p = 5.0, 16.0, 8.0
    sx, sy = mx - ux * gap, my - uy * gap
    lx, ly = mx + ux * gap, my + uy * gap
    parts = [
        f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{sx:.1f}" y2="{sy:.1f}" '
        f'stroke="{color}" stroke-width="2"/>',
        f'<line x1="{sx + px * short_p:.1f}" y1="{sy + py * short_p:.1f}" '
        f'x2="{sx - px * short_p:.1f}" y2="{sy - py * short_p:.1f}" '
        f'stroke="{color}" stroke-width="2"/>',
        f'<line x1="{lx + px * long_p:.1f}" y1="{ly + py * long_p:.1f}" '
        f'x2="{lx - px * long_p:.1f}" y2="{ly - py * long_p:.1f}" '
        f'stroke="{color}" stroke-width="3"/>',
        f'<line x1="{lx:.1f}" y1="{ly:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" '
        f'stroke="{color}" stroke-width="2"/>',
        _text_el(lx + px * (long_p + 10), ly + py * (long_p + 10), "+", color, size=13),
        _text_el(sx + px * (long_p + 10), sy + py * (long_p + 10), "−", color, size=13),
    ]
    if e.label:
        parts.append(_text_el(mx - px * (long_p + 14), my - py * (long_p + 14), e.label, "#e8eaf0"))
    return "".join(parts)


def element_svg(e):
    color = _color(e.color)
    if e.type == "rect":
        w, h = e.width or 60.0, e.height or 40.0
        out = (f'<rect x="{e.x:.1f}" y="{e.y:.1f}" width="{w:.1f}" '
               f'height="{h:.1f}" rx="4" fill="none" stroke="{color}" stroke-width="2"/>')
        if e.label:
            out += _text_el(e.x + w / 2, e.y + h / 2 + 5, e.label, "#e8eaf0")
        return out
    if e.type == "circle":
        r = e.radius or 18.0
        out = (f'<circle cx="{e.x:.1f}" cy="{e.y:.1f}" r="{r:.1f}" '
               f'fill="none" stroke="{color}" stroke-width="2"/>')
        if e.label:
            out += _text_el(e.x, e.y + r + 18, e.label, "#e8eaf0")
        return out
    if e.type in ("line", "arrow"):
        x2 = e.x2 if e.x2 is not None else e.x + (e.width or 80.0)
        y2 = e.y2 if e.y2 is not None else e.y
        marker = ' marker-end="url(#solvi-arrow)"' if e.type == "arrow" else ""
        out = (f'<line x1="{e.x:.1f}" y1="{e.y:.1f}" x2="{x2:.1f}" '
               f'y2="{y2:.1f}" stroke="{color}" stroke-width="2"{marker}/>')
        if e.label:
            out += _text_el((e.x + x2) / 2, (e.y + y2) / 2 - 8, e.label, "#e8eaf0")
        return out
    if e.type == "text":
        return _text_el(e.x, e.y, e.label or "", color, anchor="start")
    if e.type == "resistor":
        return _resistor(e, color)
    if e.type == "battery":
        return _battery(e, color)
    if e.type == "level":
        w = e.width or 120.0
        out = (f'<line x1="{e.x:.1f}" y1="{e.y:.1f}" x2="{e.x + w:.1f}" '
               f'y2="{e.y:.1f}" stroke="{color}" stroke-width="3"/>')
        if e.label:
            out += _text_el(e.x + w + 10, e.y + 5, e.label, "#e8eaf0", anchor="start", size=13)
        return out
    return None


def spec_to_svg(spec):
    body, skipped = [], []
    for e in spec.elements:
        frag = element_svg(e)
        if frag is None:
            skipped.append(e.type)
        else:
            body.append(frag)
    svg = (f'<svg viewBox="0 0 {CANVAS_W} {CANVAS_H}" width="100%" '
           'xmlns="http://www.w3.org/2000/svg" role="img" '
           'style="display:block;background:#0d0e12;">'
           "<defs>"
           '<marker id="solvi-arrow" markerWidth="10" markerHeight="8" '
           'refX="8" refY="4" orient="auto" markerUnits="userSpaceOnUse">'
           '<path d="M0,0 L9,4 L0,8 Z" fill="#e8eaf0"/>'
           "</marker></defs>" + "".join(body) + "</svg>")
    return svg, skipped


def _fb_canvas(state, model, why):
    art = TextGenerator().generate(state["question"], state["classification"], state["plan"])
    return {"artifact": art.content, "artifact_type": "html", "engine": Engine.CANVAS,
            "trace": [f"generate[canvas/{model}]: {why} → text fallback"]}


def generate_canvas(state):
    q, cls, plan = state["question"], state["classification"], state["plan"]
    model = llm.model_for(state, "generate")
    prompt = CANVAS_PROMPT.format(text=q.text, answer_key=q.answer_key,
                                  plan_json=plan.model_dump_json(indent=2), w=CANVAS_W, h=CANVAS_H)
    try:
        spec = llm.structured_call(state, "generate", prompt, CanvasSpec)
    except Exception as e:
        return _fb_canvas(state, model, f"spec call failed ({type(e).__name__})")
    svg, skipped = spec_to_svg(spec)
    drawn = len(spec.elements) - len(skipped)
    if drawn == 0:
        return _fb_canvas(state, model, "spec had no drawable elements")
    viz_panel = (f'\n<div style="padding:16px;">{svg}</div>\n'
                 f'<div style="padding:0 16px 14px;color:#9aa0ad;font-style:italic;'
                 f'text-align:center;">{escape(spec.caption)}</div>\n')
    html = render_page(q, cls, plan, viz_panel=viz_panel, engine_label="canvas")
    note = f"inline svg, {drawn} element(s)"
    if skipped:
        note += f" (skipped unknown types: {', '.join(sorted(set(skipped)))})"
    return {"artifact": html, "artifact_type": "html", "engine": Engine.CANVAS,
            "trace": [f"generate[canvas/{model}]: {note}"]}


# --------------------------------------------------------------- matter (JS) ---
MATTER_CDN = ('<script src="https://cdnjs.cloudflare.com/ajax/libs/matter-js/0.19.0/'
              'matter.min.js" '
              "onerror=\"solviCdnFail('Matter.js (physics simulation)')\"></script>")


class MatterBody(BaseModel):
    shape: Literal["circle", "rectangle"]
    x: float
    y: float
    width: Optional[float] = None
    height: Optional[float] = None
    radius: Optional[float] = None
    mass: float = 1.0
    vx: float = 0.0
    vy: float = 0.0
    color: str = "#58b8ff"
    label: str = ""


class MatterConstraint(BaseModel):
    type: str = "string"
    from_body: Optional[int] = None
    point_x: Optional[float] = None
    point_y: Optional[float] = None
    to_body: int
    length: Optional[float] = None


class MatterSpec(BaseModel):
    scenario: Literal["projectile", "collision", "pendulum", "circular", "incline", "generic"]
    bodies: list[MatterBody]
    constraints: list[MatterConstraint] = []
    gravity_y: float = 1.0
    caption: str = ""
    analytic_note: str = ""


MATTER_PROMPT = """Design a 2D physics simulation scene for this mechanics lesson.

Question: {text}
Known answer: {answer_key}
Lesson plan (JSON):
{plan_json}

The canvas is {w} wide x {h} high; x grows right, y grows DOWN.
gravity_y 1.0 is normal downward gravity; use 0.0 for horizontal collision
demos so the bodies glide.

Return:
- scenario: projectile | collision | pendulum | circular | incline | generic
- bodies: 1-5 bodies. Each: shape (circle|rectangle), x, y, radius (circles)
  or width/height (rectangles), mass, initial vx/vy, hex color, short label.
  Make sizes/masses visually proportional (e.g. the 4 kg block bigger than
  the 2 kg one) and leave room for the motion to play out on the canvas.
- constraints: strings/rods only if the scenario needs them (pendulum,
  circular). from_body null means a fixed anchor at (point_x, point_y).
- gravity_y
- caption: one line describing what the student should watch for
- analytic_note: ONE line stating the EXACT analytic result with the key
  formula (e.g. "Momentum conservation: v = (2x6)/(2+4) = 2 m/s"). This line
  is displayed as the real answer — the simulation only demonstrates it."""


def _safe_color(c, default="#58b8ff"):
    return c if c and _COLOR_RE.match(c) else default


def _clamp(v, lo, hi):
    return max(lo, min(hi, v))


def _clean_spec(spec):
    bodies = []
    for b in spec.bodies:
        bodies.append({
            "shape": b.shape,
            "x": _clamp(b.x, 0, CANVAS_W), "y": _clamp(b.y, 0, CANVAS_H),
            "radius": _clamp(b.radius or 20.0, 4, 120),
            "width": _clamp(b.width or 60.0, 4, CANVAS_W),
            "height": _clamp(b.height or 40.0, 4, CANVAS_H),
            "mass": b.mass if b.mass > 0 else 1.0,
            "vx": b.vx, "vy": b.vy, "color": _safe_color(b.color), "label": b.label[:40],
        })
    constraints = []
    for c in spec.constraints:
        if not (0 <= c.to_body < len(bodies)):
            continue
        if c.from_body is not None and not (0 <= c.from_body < len(bodies)):
            continue
        constraints.append({
            "from_body": c.from_body,
            "point_x": c.point_x if c.point_x is not None else CANVAS_W / 2,
            "point_y": c.point_y if c.point_y is not None else 40,
            "to_body": c.to_body, "length": c.length,
        })
    return {"bodies": bodies, "constraints": constraints, "gravity_y": spec.gravity_y}


MATTER_JS = """
<script>
(function () {
  var SPEC = %(spec_json)s;
  var engine = null, render = null, runner = null;

  function fail(msg) {
    var el = document.getElementById("matter-stage");
    if (el) el.innerHTML = '<div class="cdn-fail">' + msg + "</div>";
  }

  function start() {
    if (typeof Matter === "undefined") {
      fail("Could not load Matter.js. Check your connection.");
      return;
    }
    if (render) { Matter.Render.stop(render); }
    if (runner) { Matter.Runner.stop(runner); }
    if (engine) { Matter.Engine.clear(engine); }
    var stage = document.getElementById("matter-stage");
    stage.innerHTML = "";

    engine = Matter.Engine.create();
    engine.gravity.y = SPEC.gravity_y;

    var bodies = SPEC.bodies.map(function (b) {
      var body = b.shape === "circle"
        ? Matter.Bodies.circle(b.x, b.y, b.radius,
            {frictionAir: 0, friction: 0.05, restitution: 0.9,
             render: {fillStyle: b.color}})
        : Matter.Bodies.rectangle(b.x, b.y, b.width, b.height,
            {frictionAir: 0, friction: 0.05, restitution: 0.9,
             render: {fillStyle: b.color}});
      Matter.Body.setMass(body, b.mass);
      return body;
    });

    var parts = bodies.slice();
    parts.push(Matter.Bodies.rectangle(%(w)s / 2, %(h)s + 12, %(w)s + 40, 24,
      {isStatic: true, render: {fillStyle: "#262a36"}}));
    parts.push(Matter.Bodies.rectangle(-12, %(h)s / 2, 24, %(h)s,
      {isStatic: true, render: {fillStyle: "#262a36"}}));
    parts.push(Matter.Bodies.rectangle(%(w)s + 12, %(h)s / 2, 24, %(h)s,
      {isStatic: true, render: {fillStyle: "#262a36"}}));

    SPEC.constraints.forEach(function (c) {
      var opts = {bodyB: bodies[c.to_body], stiffness: 1,
                  render: {strokeStyle: "#9aa0ad"}};
      if (c.length != null) opts.length = c.length;
      if (c.from_body == null) {
        opts.pointA = {x: c.point_x, y: c.point_y};
      } else {
        opts.bodyA = bodies[c.from_body];
      }
      parts.push(Matter.Constraint.create(opts));
    });

    Matter.Composite.add(engine.world, parts);
    bodies.forEach(function (body, i) {
      Matter.Body.setVelocity(body, {x: SPEC.bodies[i].vx, y: SPEC.bodies[i].vy});
    });

    render = Matter.Render.create({
      element: stage, engine: engine,
      options: {width: %(w)s, height: %(h)s, wireframes: false,
                background: "#0d0e12"}
    });
    Matter.Render.run(render);
    runner = Matter.Runner.create();
    Matter.Runner.run(runner, engine);

    var readout = document.getElementById("matter-readout");
    Matter.Events.on(engine, "afterUpdate", function () {
      if (!readout) return;
      readout.textContent = bodies.map(function (b, i) {
        var name = SPEC.bodies[i].label || ("body " + (i + 1));
        return name + ": |v| = " + b.speed.toFixed(2);
      }).join("    ");
    });
  }

  var btn = document.getElementById("matter-restart");
  if (btn) btn.addEventListener("click", start);

  window.SolviViz = {init: start, onStage: function (i, stageId) {}};
})();
</script>
"""


def _fb_matter(state, model, why):
    art = TextGenerator().generate(state["question"], state["classification"], state["plan"])
    return {"artifact": art.content, "artifact_type": "html", "engine": Engine.MATTER,
            "trace": [f"generate[matter_js/{model}]: {why} → text fallback"]}


def generate_matter(state):
    q, cls, plan = state["question"], state["classification"], state["plan"]
    model = llm.model_for(state, "generate")
    prompt = MATTER_PROMPT.format(text=q.text, answer_key=q.answer_key,
                                  plan_json=plan.model_dump_json(indent=2), w=CANVAS_W, h=CANVAS_H)
    try:
        spec = llm.structured_call(state, "generate", prompt, MatterSpec)
    except Exception as e:
        return _fb_matter(state, model, f"spec call failed ({type(e).__name__})")
    if not spec.bodies:
        return _fb_matter(state, model, "spec has no bodies")
    scene = _clean_spec(spec)
    viz_js = MATTER_JS % {"spec_json": json.dumps(scene), "w": CANVAS_W, "h": CANVAS_H}
    analytic = escape(spec.analytic_note or plan.answer)
    viz_panel = f"""
<div style="padding:16px;">
  <div style="border:1px solid #58b8ff;border-radius:8px;padding:10px 14px;
       margin-bottom:12px;background:rgba(88,184,255,0.06);">
    <b style="color:#58b8ff;">The answer comes from the derivation:</b>
    {analytic}
  </div>
  <div id="matter-stage" style="max-width:{CANVAS_W}px;color:#9aa0ad;">
    Loading physics engine…</div>
  <div style="display:flex;align-items:center;gap:14px;margin-top:10px;
       flex-wrap:wrap;">
    <span style="color:#ffb454;font-weight:700;border:1px solid #ffb454;
         border-radius:6px;padding:2px 10px;">sim units</span>
    <span id="matter-readout" style="color:#9aa0ad;font-family:monospace;">
      </span>
    <button id="matter-restart" style="margin-left:auto;background:#1a1d26;
         color:#e8eaf0;border:1px solid #262a36;padding:6px 18px;
         border-radius:8px;cursor:pointer;">&#8635; Restart</button>
  </div>
  <div style="color:#9aa0ad;font-size:13px;margin-top:6px;">
    Readouts are in <b>sim units</b>, not SI — a game-engine demonstration of
    the relationship. The exact answer is the analytic result above.
  </div>
  <div style="color:#9aa0ad;font-style:italic;margin-top:8px;
       text-align:center;">{escape(spec.caption)}</div>
</div>
"""
    html = render_page(q, cls, plan, engine_head=MATTER_CDN, viz_panel=viz_panel,
                       viz_js=viz_js, engine_label="matter_js")
    return {"artifact": html, "artifact_type": "html", "engine": Engine.MATTER,
            "trace": [f"generate[matter_js/{model}]: {spec.scenario} scene, "
                      f"{len(scene['bodies'])} bodies, {len(scene['constraints'])} constraints"]}


## 13 · Generators: mol3d, rdkit

In [ ]:
# === mol3d + rdkit generators (inlined solvi/generate/{mol3d,rdkit}_gen.py) ===
# --------------------------------------------------------------- mol3d (JS) ---
MOL3D_CDN = ('<script src="https://3Dmol.org/build/3Dmol-min.js" '
             "onerror=\"solviCdnFail('3Dmol.js (3D molecule rendering)')\"></script>")
BOND_DISTANCE = 2.0
COORDS = {
    "octahedral": [(BOND_DISTANCE, 0, 0), (-BOND_DISTANCE, 0, 0),
                   (0, BOND_DISTANCE, 0), (0, -BOND_DISTANCE, 0),
                   (0, 0, BOND_DISTANCE), (0, 0, -BOND_DISTANCE)],
    "tetrahedral": [tuple(c * BOND_DISTANCE / math.sqrt(3) for c in corner)
                    for corner in [(1, 1, 1), (1, -1, -1), (-1, 1, -1), (-1, -1, 1)]],
    "square_planar": [(BOND_DISTANCE, 0, 0), (-BOND_DISTANCE, 0, 0),
                      (0, BOND_DISTANCE, 0), (0, -BOND_DISTANCE, 0)],
    "linear": [(BOND_DISTANCE, 0, 0), (-BOND_DISTANCE, 0, 0)],
}
ELEM_COLORS = {"N": 0x58B8FF, "O": 0xFF6B6B, "C": 0x9AA0AD, "S": 0xFFB454,
               "Cl": 0x3AD6C5, "F": 0x3AD6C5, "Br": 0xB4713D, "I": 0x8A63D2,
               "P": 0xFFB454, "H": 0xE8EAF0}


class Mol3DSpec(BaseModel):
    geometry: Literal["octahedral", "tetrahedral", "square_planar", "linear"]
    central_atom: str
    central_color: str = "#ffb454"
    ligand_atoms: list[str]
    caption: str = ""
    d_electrons: Optional[int] = None
    high_spin: Optional[bool] = None


MOL3D_PROMPT = """Design a 3D coordination-complex model for this chemistry lesson.

Question: {text}
Known answer: {answer_key}
Lesson plan (JSON):
{plan_json}

Return:
- geometry: octahedral | tetrahedral | square_planar | linear
- central_atom: element symbol of the central atom (e.g. "Co")
- central_color: hex colour for the central sphere (default "#ffb454")
- ligand_atoms: DONOR-atom element symbols, one entry per ligand, with length
  equal to the coordination number (octahedral 6, tetrahedral 4,
  square_planar 4, linear 2). E.g. six ammine ligands → ["N","N","N","N","N","N"].
- caption: one line describing what the model shows
- d_electrons: d-electron count of the central metal ION (0-10), null if not relevant
- high_spin: true/false if the spin state matters (weak vs strong field), else null"""


def _sanitize_elem(sym):
    m = _re.match(r"[A-Za-z]{1,2}", (sym or "").strip())
    return m.group(0).capitalize() if m else "X"


def _color_int(hex_color, default=0xFFB454):
    m = _re.fullmatch(r"#?([0-9a-fA-F]{6})", (hex_color or "").strip())
    return int(m.group(1), 16) if m else default


def _d_occupancy(n, high_spin):
    t2g, eg = [0, 0, 0], [0, 0]
    if high_spin:
        order = [(t2g, 0), (t2g, 1), (t2g, 2), (eg, 0), (eg, 1)] * 2
    else:
        order = [(t2g, 0), (t2g, 1), (t2g, 2)] * 2 + [(eg, 0), (eg, 1)] * 2
    for level, idx in order[:n]:
        level[idx] += 1
    return t2g, eg


def _electron_arrows(x, y, count):
    out = []
    if count >= 1:
        out.append(f'<text x="{x - 7:.0f}" y="{y - 4:.0f}" fill="#e8eaf0" '
                   f'font-size="16" text-anchor="middle">&#8593;</text>')
    if count >= 2:
        out.append(f'<text x="{x + 7:.0f}" y="{y - 4:.0f}" fill="#e8eaf0" '
                   f'font-size="16" text-anchor="middle">&#8595;</text>')
    return "".join(out)


def splitting_svg(d_electrons, high_spin):
    hs = True if high_spin is None else high_spin
    t2g, eg = _d_occupancy(d_electrons, hs)
    cfse = -0.4 * sum(t2g) + 0.6 * sum(eg)
    spin_label = "high spin" if hs else "low spin"
    w, h = 280, 340
    y_bary, y_eg, y_t2g = 170, 90, 230
    line_w, gap, x0 = 34, 46, 60
    parts = [
        f'<svg viewBox="0 0 {w} {h}" width="{w}" height="{h}" '
        'xmlns="http://www.w3.org/2000/svg" role="img" '
        'style="background:#14161d;border-radius:8px;">',
        '<text x="140" y="26" fill="#e8eaf0" font-size="14" '
        'text-anchor="middle" font-weight="600">d-orbital splitting '
        "(octahedral field)</text>",
        f'<line x1="20" y1="{y_bary}" x2="{w - 20}" y2="{y_bary}" '
        'stroke="#262a36" stroke-dasharray="5,5" stroke-width="1"/>',
        f'<text x="24" y="{y_bary - 6}" fill="#9aa0ad" font-size="11">barycentre</text>',
    ]
    for i, count in enumerate(eg):
        x = x0 + gap + i * gap
        parts.append(f'<line x1="{x - line_w / 2:.0f}" y1="{y_eg}" '
                     f'x2="{x + line_w / 2:.0f}" y2="{y_eg}" stroke="#ffb454" stroke-width="3"/>')
        parts.append(_electron_arrows(x, y_eg, count))
    parts.append(f'<text x="{w - 24}" y="{y_eg + 4}" fill="#ffb454" '
                 'font-size="12" text-anchor="end">e<tspan baseline-shift="sub" '
                 'font-size="9">g</tspan> +0.6&#916;o</text>')
    for i, count in enumerate(t2g):
        x = x0 + i * gap
        parts.append(f'<line x1="{x - line_w / 2:.0f}" y1="{y_t2g}" '
                     f'x2="{x + line_w / 2:.0f}" y2="{y_t2g}" stroke="#58b8ff" stroke-width="3"/>')
        parts.append(_electron_arrows(x, y_t2g, count))
    parts.append(f'<text x="{w - 24}" y="{y_t2g + 4}" fill="#58b8ff" '
                 'font-size="12" text-anchor="end">t<tspan baseline-shift="sub" '
                 'font-size="9">2g</tspan> &#8722;0.4&#916;o</text>')
    parts.append(f'<text x="140" y="{h - 42}" fill="#9aa0ad" font-size="12" '
                 f'text-anchor="middle">d<tspan baseline-shift="super" font-size="9">'
                 f"{d_electrons}</tspan> &#183; {spin_label}</text>")
    parts.append(f'<text x="140" y="{h - 18}" fill="#3ad6c5" font-size="14" '
                 f'text-anchor="middle" font-weight="600">CFSE = {cfse:+.1f} &#916;o</text>')
    parts.append("</svg>")
    return "".join(parts)


MOL3D_JS = """
<script>
(function () {
  var ATOMS = %(atoms_json)s;
  var BONDS = %(bonds_json)s;
  var CENTRAL = %(central_json)s;

  function fail(msg) {
    var el = document.getElementById("mol3d-viewer");
    if (el) el.innerHTML = '<div class="cdn-fail">' + msg + "</div>";
  }

  window.SolviViz = {
    init: function () {
      if (typeof $3Dmol === "undefined") {
        fail("Could not load 3Dmol.js. Check your connection.");
        return;
      }
      var element = document.getElementById("mol3d-viewer");
      element.innerHTML = "";
      var viewer = $3Dmol.createViewer(element, {backgroundColor: "#0d0e12"});
      var m = viewer.addModel();
      m.addAtoms(ATOMS);
      viewer.setStyle({}, {sphere: {radius: 0.45}});
      %(ligand_styles)s
      viewer.setStyle({elem: CENTRAL},
                      {sphere: {radius: 0.6, color: %(central_color)s}});
      BONDS.forEach(function (b) {
        viewer.addCylinder({start: {x: 0, y: 0, z: 0},
                            end: {x: b[0], y: b[1], z: b[2]},
                            radius: 0.07, color: 0x556677});
      });
      viewer.zoomTo();
      viewer.render();
      viewer.spin('y', 0.4);
    },
    onStage: function (i, stageId) {}
  };
})();
</script>
"""


def _fb_mol3d(state, model, why):
    art = TextGenerator().generate(state["question"], state["classification"], state["plan"])
    return {"artifact": art.content, "artifact_type": "html", "engine": Engine.MOL3D,
            "trace": [f"generate[mol3d_js/{model}]: {why} → text fallback"]}


def generate_mol3d(state):
    q, cls, plan = state["question"], state["classification"], state["plan"]
    model = llm.model_for(state, "generate")
    prompt = MOL3D_PROMPT.format(text=q.text, answer_key=q.answer_key,
                                 plan_json=plan.model_dump_json(indent=2))
    try:
        spec = llm.structured_call(state, "generate", prompt, Mol3DSpec)
    except Exception as e:
        return _fb_mol3d(state, model, f"spec call failed ({type(e).__name__})")
    coords = COORDS[spec.geometry]
    ligands = [_sanitize_elem(s) for s in spec.ligand_atoms][: len(coords)]
    while len(ligands) < len(coords):
        ligands.append(ligands[-1] if ligands else "X")
    central = _sanitize_elem(spec.central_atom)
    atoms = [{"elem": central, "x": 0.0, "y": 0.0, "z": 0.0}]
    for sym, (x, y, z) in zip(ligands, coords):
        atoms.append({"elem": sym, "x": round(x, 4), "y": round(y, 4), "z": round(z, 4)})
    bonds = [[round(x, 4), round(y, 4), round(z, 4)] for x, y, z in coords]
    ligand_styles = "\n      ".join(
        f'viewer.setStyle({{elem: "{sym}"}}, '
        f"{{sphere: {{radius: 0.45, color: 0x{ELEM_COLORS.get(sym, 0x58B8FF):06x}}}}});"
        for sym in sorted(set(ligands) - {central}))
    viz_js = MOL3D_JS % {"atoms_json": json.dumps(atoms), "bonds_json": json.dumps(bonds),
                         "central_json": json.dumps(central),
                         "central_color": f"0x{_color_int(spec.central_color):06x}",
                         "ligand_styles": ligand_styles}
    d_ok = spec.d_electrons is not None and 0 <= spec.d_electrons <= 10
    ledger = (f'<div style="flex:0 0 280px;display:flex;align-items:center;">'
              f"{splitting_svg(spec.d_electrons, spec.high_spin)}</div>" if d_ok else "")
    viz_panel = f"""
<div style="display:flex;flex-wrap:wrap;gap:16px;align-items:stretch;padding:16px;">
  <div id="mol3d-viewer" style="flex:1 1 380px;min-width:320px;height:360px;
       position:relative;color:#9aa0ad;">Loading 3D viewer…</div>
  {ledger}
</div>
<div style="padding:0 16px 14px;color:#9aa0ad;font-style:italic;
     text-align:center;">{escape(spec.caption)}</div>
"""
    html = render_page(q, cls, plan, engine_head=MOL3D_CDN, viz_panel=viz_panel,
                       viz_js=viz_js, engine_label="mol3d_js")
    ledger_note = "with CFSE ledger" if d_ok else "no d-electron ledger"
    return {"artifact": html, "artifact_type": "html", "engine": Engine.MOL3D,
            "trace": [f"generate[mol3d_js/{model}]: {spec.geometry} "
                      f"{central}({len(ligands)} ligands), {ledger_note}"]}


# --------------------------------------------------------------- rdkit (JS) ---
RDKIT_CDN = ('<script src="https://cdn.jsdelivr.net/npm/@rdkit/rdkit@2024.9.5/'
             'dist/RDKit_minimal.js" '
             "onerror=\"solviCdnFail('RDKit.js (molecule rendering)')\"></script>")


class HighlightGroup(BaseModel):
    stage_id: str
    atom_indices: list[int]
    caption: str = ""


class HighlightSpec(BaseModel):
    highlights: list[HighlightGroup] = []


HIGHLIGHT_PROMPT = """You annotate a 2D molecule drawing for a chemistry lesson.

Molecule SMILES: {smiles}
Atom indices (index:symbol): {atom_map}
Lesson stage ids: {stage_ids}
Lesson plan (JSON):
{plan_json}

For each lesson stage where highlighting specific atoms would TEACH something
(the chiral centre at the 'trap' stage, the double bond at a mechanism step,
the functional group at 'concept'...), return one highlight group:
  {{"stage_id": <stage id from the list>, "atom_indices": [<indices from the
  map above>], "caption": "one short line shown under the drawing"}}
Only include stages that genuinely benefit. Atom indices MUST come from the
map above."""


def _get_highlights(state, smiles, num_atoms):
    from rdkit import Chem
    mol = Chem.MolFromSmiles(smiles)
    atom_map = ", ".join(f"{a.GetIdx()}:{a.GetSymbol()}" for a in mol.GetAtoms())
    stage_ids = [s["id"] for s in stages_from_plan(state["plan"])]
    prompt = HIGHLIGHT_PROMPT.format(smiles=smiles, atom_map=atom_map,
                                     stage_ids=", ".join(stage_ids),
                                     plan_json=state["plan"].model_dump_json(indent=2))
    try:
        spec = llm.structured_call(state, "generate", prompt, HighlightSpec)
    except Exception as e:
        return [], f"highlights skipped ({type(e).__name__})"
    groups, dropped = [], 0
    for g in spec.highlights:
        if g.atom_indices and all(0 <= i < num_atoms for i in g.atom_indices):
            groups.append({"stage_id": g.stage_id, "atom_indices": g.atom_indices,
                           "caption": g.caption})
        else:
            dropped += 1
    note = f"{len(groups)} highlight group(s)"
    if dropped:
        note += f" ({dropped} dropped: bad atom indices)"
    return groups, note


RDKIT_PANEL = """
<div id="rdkit-panel" style="padding:18px;min-height:240px;display:flex;
     align-items:center;justify-content:center;">Loading molecule…</div>
<div id="rdkit-caption" style="padding:0 18px 14px;color:#9aa0ad;
     font-style:italic;text-align:center;"></div>
"""

RDKIT_JS = """
<script>
(function () {
  var SMILES = %(smiles_json)s;
  var GROUPS = %(groups_json)s;
  var SVGS = {};
  var BASE = "";

  function panel() { return document.getElementById("rdkit-panel"); }
  function fail(msg) {
    var p = panel();
    if (p) p.innerHTML = '<div class="cdn-fail">' + msg + "</div>";
  }
  function draw(stageId) {
    var p = panel();
    if (!p || !BASE) return;
    var hit = SVGS[stageId];
    p.innerHTML = hit ? hit.svg : BASE;
    var cap = document.getElementById("rdkit-caption");
    if (cap) cap.textContent = hit ? hit.caption : "";
  }

  window.SolviViz = {
    init: async function () {
      if (typeof initRDKitModule === "undefined") {
        fail("Could not load RDKit.js. Check your connection.");
        return;
      }
      var RDKit;
      try {
        RDKit = await initRDKitModule();
      } catch (e) {
        fail("Could not load RDKit.js. Check your connection.");
        return;
      }
      var mol = RDKit.get_mol(SMILES);
      if (!mol || !mol.is_valid()) {
        if (mol) mol.delete();
        fail("The molecule could not be rendered.");
        return;
      }
      var base = {
        width: 760, height: 300, bondLineWidth: 2,
        backgroundColour: [0.98, 0.98, 0.99, 1.0],
        addStereoAnnotation: true
      };
      BASE = mol.get_svg_with_highlights(JSON.stringify(base));
      GROUPS.forEach(function (g) {
        var details = Object.assign({}, base, {
          atoms: g.atom_indices,
          highlightColour: [0.72, 0.61, 1.0],
          legend: g.caption || ""
        });
        SVGS[g.stage_id] = {
          svg: mol.get_svg_with_highlights(JSON.stringify(details)),
          caption: g.caption || ""
        };
      });
      mol.delete();
      draw("concept");
    },
    onStage: function (i, stageId) { draw(stageId); }
  };
})();
</script>
"""


def _fb_rdkit(state, model, why):
    art = TextGenerator().generate(state["question"], state["classification"], state["plan"])
    return {"artifact": art.content, "artifact_type": "html", "engine": Engine.RDKIT,
            "flags": ["invalid_smiles"],
            "trace": [f"generate[rdkit_js/{model}]: {why} → text fallback"]}


def generate_rdkit(state):
    from rdkit import Chem
    cls = state["classification"]
    model = llm.model_for(state, "generate")
    if not cls.smiles:
        return _fb_rdkit(state, model, "no SMILES on classification")
    mol = Chem.MolFromSmiles(cls.smiles)
    if mol is None:
        return _fb_rdkit(state, model, f"SMILES failed to parse: {cls.smiles!r}")
    groups, note = _get_highlights(state, cls.smiles, mol.GetNumAtoms())
    viz_js = RDKIT_JS % {"smiles_json": json.dumps(cls.smiles), "groups_json": json.dumps(groups)}
    html = render_page(state["question"], cls, state["plan"], engine_head=RDKIT_CDN,
                       viz_panel=RDKIT_PANEL, viz_js=viz_js, engine_label="rdkit_js")
    return {"artifact": html, "artifact_type": "html", "engine": Engine.RDKIT,
            "trace": [f"generate[rdkit_js/{model}]: html, {note}"]}


## 14 · Generator: manim (AST gate + render)

In [ ]:
# === manim generator (inlined solvi/generate/manim_gen.py) ===
# Emits a Manim scene from the plan, AST-gates it, renders at -qm, and on any
# failure (no manim/LaTeX/ffmpeg, bad code twice, render error) degrades to the
# text artifact with 'manim_render_failed'. ALLOWED_IMPORTS is renamed here so it
# does not clobber the verifier's sandbox whitelist.
MANIM_ALLOWED_IMPORTS = {"manim", "math", "numpy"}
FORBIDDEN_CALLS = {"open", "exec", "eval", "compile", "__import__", "input",
                   "getattr", "setattr", "vars", "globals", "locals", "breakpoint"}

MANIM_PROMPT = """Write a Manim Community v0.20 Scene that teaches this lesson.

Lesson plan (JSON):
{plan_json}

Constraints:
- class name: GeneratedScene(Scene)
- Dark background: self.camera.background_color = "#0d0e12"
- Palette: AMBER="#ffb454" BLUE="#58b8ff" TEAL="#3ad6c5" RED="#ff6b6b" DIM="#9aa0ad"
- Use MathTex for all mathematics (LaTeX is available).
- Use Text(font="sans-serif") for prose. Keep prose lines under 70 chars —
  WRAP LONG LINES WITH \\n or they overflow the frame.
- Structure the scene to follow the plan: concept primer → approach → steps → answer → trap.
- Use ReplacementTransform to MORPH equations (this is the 3B1B signature move).
{pacing_block}
- Do NOT overlap elements. Use .to_edge(), .next_to(), and FadeOut old mobjects
  before introducing new ones in the same region.
- Imports: ONLY from manim, math, numpy.
Return ONLY python code, no markdown fences, no commentary.
{revision_block}"""

DEFAULT_PACING = "- Pace it: self.wait(0.8) after each beat. Target 45-70 seconds."

NARRATED_PACING_TEMPLATE = """- NARRATED TIMING CONTRACT — this scene is dubbed with pre-recorded narration:
  * The scene has exactly {n} beats. The FIRST line of construct() must be
    self.next_section("beat_0"); at the start of every later beat call
    self.next_section("beat_<i>"). Exactly {n} sections, in order, no extras.
  * Each beat's animations plus self.wait() calls MUST total at least its
    minimum duration below. End every beat with a self.wait() sized so the
    beat comfortably exceeds its minimum (~0.5s margin).
  * Minimum durations (seconds):
{duration_table}
  * Each beat's visuals must show what its narration says:
{narration_table}"""

REVISION_TEMPLATE = """
A previous version of this scene had problems. Here is that code:

--- previous code ---
{prior_code}
--- end previous code ---

Problems that MUST be fixed this time:
{complaints}

Rewrite the scene fixing every problem above. Keep what already worked."""


def strip_code_fences(text):
    t = text.strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1:
            t = t[first_newline + 1:]
        if t.rstrip().endswith("```"):
            t = t.rstrip()[:-3]
    return t.strip()


def validate_scene_code(code):
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"syntax error: {e}"
    has_scene = False
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in MANIM_ALLOWED_IMPORTS:
                    return f"forbidden import: {alias.name}"
        elif isinstance(node, ast.ImportFrom):
            if (node.module or "").split(".")[0] not in MANIM_ALLOWED_IMPORTS:
                return f"forbidden import: from {node.module}"
        elif isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            return f"forbidden dunder attribute: {node.attr}"
        elif isinstance(node, ast.Call):
            fn = node.func
            if isinstance(fn, ast.Name) and fn.id in FORBIDDEN_CALLS:
                return f"forbidden call: {fn.id}()"
        elif isinstance(node, ast.ClassDef) and node.name == "GeneratedScene":
            has_scene = True
    if not has_scene:
        return "code does not define class GeneratedScene"
    return None


def render_scene(code, workdir, timeout, save_sections=False):
    workdir.mkdir(parents=True, exist_ok=True)
    scene_path = workdir / "scene.py"
    scene_path.write_text(code)
    cmd = [sys.executable, "-m", "manim", "-qm", "--format", "mp4",
           "--media_dir", str(workdir / "media"), "-o", "output",
           *(["--save_sections"] if save_sections else []),
           str(scene_path), "GeneratedScene"]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None, f"render timed out after {timeout}s"
    except FileNotFoundError as e:
        return None, f"manim not available: {e}"
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "")[-1500:]
        return None, f"manim exited {proc.returncode}: {tail}"
    hits = list((workdir / "media").rglob("output.mp4"))
    if not hits:
        return None, "render reported success but no output.mp4 found"
    return hits[0], ""


def _pacing_block(state):
    track = state.get("audio_track")
    script = state.get("narration_script")
    if not track or not script:
        return DEFAULT_PACING
    durations = {s.index: s.duration for s in track.segments}
    duration_table = "\n".join(
        f"      beat_{b.index}: {durations.get(b.index, 0.0) + 0.5:.1f}" for b in script.beats)
    narration_table = "\n".join(f'      beat_{b.index}: "{b.text}"' for b in script.beats)
    return NARRATED_PACING_TEMPLATE.format(n=len(script.beats),
                                           duration_table=duration_table,
                                           narration_table=narration_table)


def _emit_code(state, complaints, prior_code):
    plan = state["plan"]
    revision_block = ""
    if complaints and prior_code:
        revision_block = REVISION_TEMPLATE.format(
            prior_code=prior_code, complaints="\n".join(f"- {c}" for c in complaints))
    prompt = MANIM_PROMPT.format(plan_json=plan.model_dump_json(indent=2),
                                 pacing_block=_pacing_block(state), revision_block=revision_block)
    return strip_code_fences(llm.text_call(state, "generate", prompt))


def _fb_manim(state, attempts, why):
    art = TextGenerator().generate(state["question"], state["classification"], state["plan"])
    return {"artifact": art.content, "artifact_type": "html", "engine": Engine.MANIM,
            "render_attempts": attempts, "layout_problems": [],
            "flags": ["manim_render_failed"],
            "trace": [f"generate[manim]: render failed → text fallback ({why[:120]})"]}


def _find_sections_file(video, workdir):
    candidate = video.parent / "sections" / f"{video.stem}.json"
    if candidate.exists():
        return str(candidate)
    hits = [p for p in (workdir / "media").rglob(f"{video.stem}.json")
            if p.parent.name == "sections"]
    return str(hits[0]) if hits else None


def generate_manim(state):
    attempts = state.get("render_attempts", 0) + 1
    complaints = list(state.get("layout_problems") or [])
    prior_code = state.get("generated_code")
    model = llm.model_for(state, "generate")
    narrated = bool(state.get("audio_track"))
    code = _emit_code(state, complaints, prior_code)
    err = validate_scene_code(code)
    if err:
        code = _emit_code(state, [f"the code was rejected: {err}"], code)
        err = validate_scene_code(code)
        if err:
            return _fb_manim(state, attempts, f"code invalid twice: {err}")
    q = state["question"]
    workdir = settings().artifact_dir / q.id / f"render_{attempts}"
    video, detail = render_scene(code, workdir, settings().render_timeout, save_sections=narrated)
    if video is None and "manim exited" in detail:
        code2 = _emit_code(state, [f"rendering failed with: {detail}"], code)
        if validate_scene_code(code2) is None:
            code = code2
            video, detail = render_scene(code, workdir, settings().render_timeout,
                                         save_sections=narrated)
    if video is None:
        return _fb_manim(state, attempts, detail)
    return {"artifact": str(video), "artifact_type": "video", "engine": Engine.MANIM,
            "generated_code": code, "render_attempts": attempts,
            "sections_file": _find_sections_file(video, workdir) if narrated else None,
            "trace": [f"generate[manim/{model}]: rendered {video.name} (attempt {attempts})"]}


## 15 · Narration: script, TTS audio, mux, frame-check

In [ ]:
# === narration / audio / mux / frame_check (inlined solvi/generate/*) ===
# The audio-first dubbing chain (manim path only). Every failure degrades to a
# silent/text artifact — never raises. Reached only when RUN['narration']=True
# and the question routes to manim.
import base64
import re
from difflib import SequenceMatcher


def duration_seconds(media):
    proc = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(media)],
        capture_output=True, text=True, timeout=30)
    return float(proc.stdout.strip())


# --------------------------------------------------------- narration_gen.py ---
class ScriptReview(BaseModel):
    ok: bool
    problems: list[str] = Field(default_factory=list)


LANGUAGE_RULES = {
    "hinglish": ("Hinglish written entirely in LATIN script (roman Hindi — absolutely "
                 "no Devanagari characters). Hindi connectors and encouragement "
                 "('dekho', 'ab yahan', 'toh', 'samjhe?'), English for ALL technical "
                 "terms, formulas and numbers. Example: 'Dekho beta, yahan hume moment "
                 "of inertia nikalna hai — formula hai one by two M R square.'"),
    "english": ("Simple, clear Indian English. Short sentences a school student "
                "follows easily. No slang, no complicated vocabulary."),
    "hindi": ("Conversational Hindi, but romanized in LATIN script (no Devanagari "
              "characters) so the voice reads it reliably. Technical terms, "
              "formulas and numbers stay in English."),
}

SCRIPT_PROMPT = """You are an experienced Indian teacher recording the voice-over for a short
solution video. Write the spoken narration.

Verified lesson plan (JSON):
{plan_json}

The video has exactly {n} beats, in this order:
{beat_outline}

Draft caption per beat (improve on these, do not just copy):
{draft_lines}

Write exactly one narration beat per video beat, same order, index 0 to {n_minus_1}.

LANGUAGE: {language_rules}

SPOKEN MATH — a text-to-speech voice reads this aloud, so:
- NEVER write LaTeX, backslashes, ^, _, {{ }}, or symbols like ∫ Σ √ π ².
- Write every expression the way a teacher SAYS it:
  "\\frac{{13}}{{32}} M R^2"  → "thirteen by thirty-two M R square"
  "x^2 + 3x - 4 = 0"         → "x square plus three x minus four equals zero"
  "∫₀¹ x dx"                 → "integral from zero to one of x d x"
- Speak units and symbols naturally ("metre per second square", "capital R").

TONE: warm, patient, speaking to one student. Short sentences, like a teacher
at the board — not an essay. 2-4 sentences per beat (about 8-15 seconds of
speech). The answer beat must state the final answer explicitly.
{revision_block}"""

REVISION_TEMPLATE_N = """
A previous version of this script had problems. Here is that script:

--- previous script ---
{prior_script}
--- end previous script ---

Problems that MUST be fixed this time:
{complaints}

Rewrite the script fixing every problem above. Keep what already worked."""

JUDGE_PROMPT = """You are reviewing a narration script before it is recorded by text-to-speech.
Your job is to catch defects that would make the RECORDING wrong or unusable —
nothing else. This is a blocking gate, not an editorial review.

Ground truth — the verified solution plan (JSON):
{plan_json}

Expected beat structure ({n} beats):
{beat_outline}

Narration script ({language}), one beat per line:
{numbered_script}

Report ONLY these blocking violations, and ONLY if you can copy the exact
offending words from the script. Format each problem as:
"beat k: '<exact quote from that beat>' — <what is wrong>".
1. WRONG MATH: a number, expression, or the final answer that CONTRADICTS the
   plan. Re-read the quoted text carefully before claiming this — if the
   quoted words state the correct relation, it is NOT a violation.
2. UNSPEAKABLE NOTATION: LaTeX, backslashes, ^ _ {{ }} or symbols like
   ∫ Σ √ π² that a voice cannot read aloud.
3. WRONG SCRIPT: any Devanagari character (the script must be Latin-only).
4. WRONG LANGUAGE: the beat is not {language}. Expected style: {language_rules}

Calibration — do NOT report:
- stylistic or phrasing preferences, pacing, or "could also mention ..." —
  missing extra detail is never a violation
- a beat that first describes the common MISTAKE and then gives the correct
  version (that is deliberate teaching, especially in the trap beat)
- romanized Hindi spelling variants (hume/humein, nikalna/nikaalna)
- numbers spoken in words vs digits
Return ok=true with an empty problems list unless a quoted, verified
violation of rules 1-4 exists."""


def beat_outline(plan):
    beats = ["concept primer", "approach"]
    beats += [f"step {i + 1}: {s.title}" for i, s in enumerate(plan.steps)]
    beats += ["answer", "trap (common mistake)"]
    return beats


def _outline_text(outline):
    return "\n".join(f"  beat_{i}: {b}" for i, b in enumerate(outline))


def _script_lines(script):
    return "\n".join(f"beat_{b.index}: {b.text}" for b in script.beats)


def gen_script(state):
    plan = state["plan"]
    cfg = _run_config(state)
    attempts = state.get("narration_attempts", 0) + 1
    model = llm.model_for(state, "narrate")
    outline = beat_outline(plan)
    complaints = list(state.get("narration_problems") or [])
    prior = state.get("narration_script")
    revision_block = ""
    if complaints and prior:
        revision_block = REVISION_TEMPLATE_N.format(
            prior_script=_script_lines(prior),
            complaints="\n".join(f"- {c}" for c in complaints))
    prompt = SCRIPT_PROMPT.format(
        plan_json=plan.model_dump_json(indent=2), n=len(outline), n_minus_1=len(outline) - 1,
        beat_outline=_outline_text(outline),
        draft_lines="\n".join(f"- {line}" for line in plan.narration) or "(none)",
        language_rules=LANGUAGE_RULES[cfg.narration_language], revision_block=revision_block)
    try:
        script = llm.structured_call(state, "narrate", prompt, NarrationScript)
    except Exception as e:
        return {"narration_script": None, "narration_problems": [], "narration_attempts": attempts,
                "flags": ["narration_failed"],
                "trace": [f"gen_script[{model}]: error — {type(e).__name__}: {e} → silent video"]}
    script = NarrationScript(language=cfg.narration_language,
                             beats=[b.model_copy(update={"index": i})
                                    for i, b in enumerate(script.beats)])
    return {"narration_script": script, "narration_attempts": attempts,
            "trace": [f"gen_script[{model}]: {len(script.beats)} beats, "
                      f"{cfg.narration_language} (attempt {attempts})"]}


def script_check(state):
    script = state.get("narration_script")
    if script is None:
        return {"narration_problems": [], "trace": ["script_check: skipped (no script)"]}
    plan = state["plan"]
    cfg = _run_config(state)
    model = llm.model_for(state, "script_check")
    outline = beat_outline(plan)
    prompt = JUDGE_PROMPT.format(plan_json=plan.model_dump_json(indent=2), n=len(outline),
                                 beat_outline=_outline_text(outline), language=script.language,
                                 numbered_script=_script_lines(script),
                                 language_rules=LANGUAGE_RULES[script.language])
    try:
        review = llm.structured_call(state, "script_check", prompt, ScriptReview)
    except Exception as e:
        return {"narration_problems": [], "flags": ["script_check_error"],
                "trace": [f"script_check[{model}]: error — {type(e).__name__}: {e} → unjudged"]}
    if review.problems:
        if state.get("narration_attempts", 0) >= cfg.max_audio_attempts:
            return {"narration_script": None, "narration_problems": [], "flags": ["narration_failed"],
                    "trace": [f"script_check[{model}]: FAILED at attempt cap — {review.problems}"
                              " → silent video"]}
        return {"narration_problems": review.problems,
                "trace": [f"script_check[{model}]: FAILED — {review.problems}"]}
    return {"narration_problems": [], "trace": [f"script_check[{model}]: ok"]}


# ------------------------------------------------------------- audio_gen.py ---
TTS_INSTRUCTIONS = {
    "hinglish": ("Warm, patient Indian teacher explaining to a student at the board. "
                 "Natural Hinglish code-switching; Hindi connectors relaxed and "
                 "conversational, English technical terms and numbers crisp and clear. "
                 "Moderate pace."),
    "english": ("Warm, patient Indian teacher explaining to a student at the board. "
                "Clear simple English, moderate pace, numbers and formulas spoken crisply."),
    "hindi": ("Warm, patient Indian teacher explaining to a student in "
              "conversational Hindi; English technical terms and numbers crisp and "
              "clear. Moderate pace."),
}
MATCH_THRESHOLD = {"english": 0.80, "hinglish": 0.60, "hindi": 0.55}
TRANSCRIBE_PROMPT = {
    "english": None,
    "hinglish": ("Hinglish lecture transcribed in Latin script (roman Hindi), "
                 "e.g.: Dekho beta, ab hum formula use karenge."),
    "hindi": ("Hindi lecture transcribed in Latin script (roman Hindi), "
              "e.g.: Dekho beta, ab hum formula use karenge."),
}
MIN_COMPARABLE_CHARS = 10
DEVANAGARI_HEAVY = 0.3
_DEV_INDEPENDENT = {"अ": "a", "आ": "aa", "इ": "i", "ई": "i", "उ": "u", "ऊ": "u", "ऋ": "ri",
                    "ए": "e", "ऐ": "ai", "ओ": "o", "औ": "au", "ऑ": "o"}
_DEV_CONSONANT = {"क": "k", "ख": "kh", "ग": "g", "घ": "gh", "ङ": "n",
                  "च": "ch", "छ": "chh", "ज": "j", "झ": "jh", "ञ": "n",
                  "ट": "t", "ठ": "th", "ड": "d", "ढ": "dh", "ण": "n",
                  "त": "t", "थ": "th", "द": "d", "ध": "dh", "न": "n",
                  "प": "p", "फ": "f", "ब": "b", "भ": "bh", "म": "m",
                  "य": "y", "र": "r", "ल": "l", "व": "v",
                  "श": "sh", "ष": "sh", "स": "s", "ह": "h",
                  "क़": "q", "ख़": "kh", "ग़": "g", "ज़": "z", "फ़": "f", "ड़": "r", "ढ़": "rh"}
_DEV_MATRA = {"ा": "aa", "ि": "i", "ी": "i", "ु": "u", "ू": "u", "ृ": "ri",
              "े": "e", "ै": "ai", "ो": "o", "ौ": "au", "ॉ": "o"}
_DEV_SIGN = {"ं": "n", "ँ": "n", "ः": "", "़": "", "्": "", "।": " ", "॥": " "}


def devanagari_ratio(text):
    if not text:
        return 0.0
    dev = sum(1 for ch in text if "ऀ" <= ch <= "ॿ")
    return dev / len(text)


def transliterate_devanagari(text):
    out, chars = [], list(text)
    for i, ch in enumerate(chars):
        if ch in _DEV_CONSONANT:
            out.append(_DEV_CONSONANT[ch])
            nxt = chars[i + 1] if i + 1 < len(chars) else ""
            if nxt not in _DEV_MATRA and nxt != "्":
                out.append("a")
        elif ch in _DEV_MATRA:
            out.append(_DEV_MATRA[ch])
        elif ch in _DEV_INDEPENDENT:
            out.append(_DEV_INDEPENDENT[ch])
        elif ch in _DEV_SIGN:
            out.append(_DEV_SIGN[ch])
        else:
            out.append(ch)
    return "".join(out)


def normalize_for_match(text):
    text = transliterate_devanagari(text).lower()
    text = text.encode("ascii", errors="ignore").decode()
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def transcript_match(script_text, transcript):
    a = normalize_for_match(script_text)
    b = normalize_for_match(transcript)
    if len(a) < MIN_COMPARABLE_CHARS or len(b) < MIN_COMPARABLE_CHARS:
        return None
    return SequenceMatcher(None, a, b, autojunk=False).ratio()


def gen_audio(state):
    script = state.get("narration_script")
    if script is None:
        return {"audio_track": None, "trace": ["gen_audio: skipped (no script)"]}
    cfg = _run_config(state)
    q = state["question"]
    attempt = state.get("narration_attempts", 1)
    outdir = settings().artifact_dir / q.id / "audio" / f"attempt_{attempt}"
    model = llm.media_model_for(state, "tts")
    instructions = TTS_INSTRUCTIONS[script.language]
    try:
        outdir.mkdir(parents=True, exist_ok=True)
        segments = []
        for beat in script.beats:
            audio = llm.speech_call(state, "tts", beat.text,
                                    voice=cfg.narration_voice, instructions=instructions)
            path = outdir / f"beat_{beat.index}.mp3"
            path.write_bytes(audio)
            segments.append(AudioSegment(index=beat.index, path=str(path),
                                         duration=duration_seconds(path)))
    except Exception as e:
        return {"audio_track": None, "flags": ["tts_error", "narration_failed"],
                "trace": [f"gen_audio[{model}]: error — {type(e).__name__}: {e} → silent video"]}
    track = AudioTrack(voice=cfg.narration_voice, segments=segments,
                       total=sum(s.duration for s in segments))
    return {"audio_track": track,
            "trace": [f"gen_audio[{model}]: {len(segments)} segments, "
                      f"{track.total:.1f}s total (voice={track.voice})"]}


def audio_check(state):
    track = state.get("audio_track")
    script = state.get("narration_script")
    if track is None or script is None:
        return {"narration_problems": [], "trace": ["audio_check: skipped (no audio)"]}
    cfg = _run_config(state)
    model = llm.media_model_for(state, "stt")
    beat_text = {b.index: b.text for b in script.beats}
    language = "en" if script.language == "english" else None
    stt_prompt = TRANSCRIBE_PROMPT[script.language]
    checked, problems, unverifiable = [], [], 0
    try:
        for seg in track.segments:
            kwargs = {"prompt": stt_prompt} if stt_prompt else {}
            transcript = llm.transcribe_call(state, "stt", seg.path, language=language, **kwargs)
            ratio = transcript_match(beat_text.get(seg.index, ""), transcript)
            checked.append(seg.model_copy(update={"transcript": transcript, "match": ratio}))
            if ratio is None:
                unverifiable += 1
            elif ratio < MATCH_THRESHOLD[script.language]:
                if devanagari_ratio(transcript) >= DEVANAGARI_HEAVY:
                    unverifiable += 1
                else:
                    problems.append(
                        f"beat {seg.index}: audio transcribed as {transcript!r} vs "
                        f"script {beat_text.get(seg.index, '')!r} (ratio {ratio:.2f})"
                        " — rewrite this beat with simpler, more phonetic wording")
    except Exception as e:
        return {"narration_problems": [], "flags": ["stt_error", "unverified_narration"],
                "trace": [f"audio_check[{model}]: error — {type(e).__name__}: {e} → unverified"]}
    track = track.model_copy(update={"segments": checked})
    if problems:
        if state.get("narration_attempts", 0) >= cfg.max_audio_attempts:
            return {"audio_track": None, "narration_problems": [], "flags": ["narration_failed"],
                    "trace": [f"audio_check[{model}]: FAILED at attempt cap — "
                              f"{problems} → silent video"]}
        return {"audio_track": track, "narration_problems": problems,
                "trace": [f"audio_check[{model}]: FAILED — {problems}"]}
    note = f", {unverifiable} unverifiable" if unverifiable else ""
    flags = ["unverified_narration"] if unverifiable else []
    return {"audio_track": track, "narration_problems": [], "flags": flags,
            "trace": [f"audio_check[{model}]: ok ({len(track.segments)} segments{note})"]}


# ------------------------------------------------------------------ mux.py ---
SEGMENT_GAP = 0.15
FALLBACK_LEAD = 0.5
FALLBACK_GAP = 0.35
TAIL_MARGIN = 0.5
MUX_TIMEOUT = 300.0


def section_starts(sections, n_beats):
    names = [s.get("name") for s in sections]
    if names != [f"beat_{i}" for i in range(n_beats)]:
        return None
    starts, t = [], 0.0
    for s in sections:
        starts.append(t)
        try:
            t += float(s["duration"])
        except (KeyError, TypeError, ValueError):
            return None
    return starts


def plan_placement(starts, durations):
    if starts is None:
        places, t = [], FALLBACK_LEAD
        for d in durations:
            places.append(t)
            t += d + FALLBACK_GAP
        return places
    places, prev_end = [], -SEGMENT_GAP
    for start, dur in zip(starts, durations):
        place = max(start, prev_end + SEGMENT_GAP)
        places.append(place)
        prev_end = place + dur
    return places


def build_mux_command(video, segment_paths, places, pad, out):
    cmd = ["ffmpeg", "-y", "-i", video]
    for p in segment_paths:
        cmd += ["-i", p]
    n = len(segment_paths)
    filters = []
    for i in range(n):
        ms = round(places[i] * 1000)
        filters.append(f"[{i + 1}:a]adelay={ms}:all=1[a{i}]")
    mix_in = "".join(f"[a{i}]" for i in range(n))
    filters.append(f"{mix_in}amix=inputs={n}:normalize=0[aout]")
    if pad > 0:
        filters.insert(0, f"[0:v]tpad=stop_mode=clone:stop_duration={pad:.3f}[v]")
        video_maps = ["-map", "[v]", "-c:v", "libx264", "-preset", "veryfast", "-crf", "20"]
    else:
        video_maps = ["-map", "0:v", "-c:v", "copy"]
    cmd += ["-filter_complex", ";".join(filters)]
    cmd += video_maps
    cmd += ["-map", "[aout]", "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", out]
    return cmd


def mux_audio(state):
    track = state.get("audio_track")
    if state.get("artifact_type") != "video" or track is None:
        return {"trace": ["mux_audio: skipped (no audio to mux)"]}
    video = state["artifact"]
    segments = sorted(track.segments, key=lambda s: s.index)
    try:
        starts = None
        if state.get("sections_file"):
            sections = json.loads(Path(state["sections_file"]).read_text())
            starts = section_starts(sections, len(segments))
        places = plan_placement(starts, [s.duration for s in segments])
        video_dur = duration_seconds(video)
        tail = places[-1] + segments[-1].duration + TAIL_MARGIN
        pad = max(0.0, tail - video_dur)
        out = settings().artifact_dir / state["question"].id / "final.mp4"
        out.parent.mkdir(parents=True, exist_ok=True)
        cmd = build_mux_command(video, [s.path for s in segments], places, pad, str(out))
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=MUX_TIMEOUT)
        if proc.returncode != 0 or not out.exists():
            raise RuntimeError(f"ffmpeg exited {proc.returncode}: {(proc.stderr or '')[-500:]}")
    except Exception as e:
        return {"flags": ["mux_failed", "narration_failed"],
                "trace": [f"mux_audio: error — {type(e).__name__}: {e} → silent video"]}
    mode = "sections" if starts is not None else "fallback layout"
    return {"artifact": str(out),
            "trace": [f"mux_audio: {len(segments)} segments via {mode}"
                      + (f", padded {pad:.1f}s" if pad > 0 else "")]}


# ---------------------------------------------------------- frame_check.py ---
class FrameCheckResult(BaseModel):
    problems: list[str] = Field(default_factory=list)


FRAME_CHECK_PROMPT = """This image is a contact sheet of {n} frames sampled evenly from an
educational maths/science animation, tiled left-to-right, top-to-bottom
(frame 1 is top-left). Inspect each frame for LAYOUT defects only:

- text overlapping other text or diagrams (illegible)
- text cut off at the frame edge or clearly running off-frame
- elements stacked so the frame is unreadable

Report each defect as one short specific string, e.g.
"frames 3-4: title overlaps the equation in the centre".

Calibration — be strict about what counts:
- Only report a defect you can clearly SEE in a specific frame.
- Two sentences printed on top of each other (interleaved, jumbled,
  double-struck letters) IS a defect — the worst kind.
- A sentence whose start or end is missing at a tile's edge IS a defect
  (it ran off-frame).
- Text that is merely close to other text but each line readable is NOT
  a defect. Text mid-animation (partially drawn/fading) is NOT a defect.
- Do NOT report stylistic opinions, pacing, or content correctness.
- The grey borders separating the tiles are part of the sheet, not defects.
If every frame is clean, return an empty problems list."""


def extract_frames(video_path, n=6):
    video = Path(video_path)
    duration = duration_seconds(video)
    cols = 3
    rows = max(1, (n + cols - 1) // cols)
    with tempfile.TemporaryDirectory() as tmp:
        frame_paths = []
        for i in range(n):
            t = min(duration * (i + 0.95) / n, duration - 0.05)
            out = Path(tmp) / f"frame_{i}.jpg"
            subprocess.run(["ffmpeg", "-y", "-ss", f"{t:.2f}", "-i", str(video),
                            "-frames:v", "1", "-q:v", "3", str(out)],
                           capture_output=True, timeout=60)
            if out.exists():
                frame_paths.append(out)
        if not frame_paths:
            return []
        while len(frame_paths) < cols * rows:
            frame_paths.append(frame_paths[-1])
        sheet = Path(tmp) / "sheet.jpg"
        inputs = []
        for p in frame_paths:
            inputs += ["-i", str(p)]
        filters = [f"[{i}]scale=640:-1,pad=652:372:6:6:color=#666666[s{i}]"
                   for i in range(len(frame_paths))]
        for r in range(rows):
            trio = "".join(f"[s{r * cols + c}]" for c in range(cols))
            filters.append(f"{trio}hstack={cols}[row{r}]")
        if rows > 1:
            filters.append("".join(f"[row{r}]" for r in range(rows)) + f"vstack={rows}[out]")
            out_label = "[out]"
        else:
            out_label = "[row0]"
        subprocess.run(["ffmpeg", "-y", *inputs, "-filter_complex", ";".join(filters),
                        "-map", out_label, "-q:v", "3", str(sheet)],
                       capture_output=True, timeout=60)
        if not sheet.exists():
            return []
        return [base64.b64encode(sheet.read_bytes()).decode()]


def vision_check(state, frames, n_frames=6):
    images = [{"kind": "base64", "data": f, "mime_type": "image/jpeg"} for f in frames]
    result = llm.structured_call(state, "frame_check", FRAME_CHECK_PROMPT.format(n=n_frames),
                                 FrameCheckResult, images=images)
    return result.problems


def frame_check(state):
    if state.get("artifact_type") != "video":
        return {"layout_problems": [], "trace": ["frame_check: skipped (no video)"]}
    model = llm.model_for(state, "frame_check")
    try:
        frames = extract_frames(state["artifact"])
        if not frames:
            raise RuntimeError("no frames extracted")
        problems = vision_check(state, frames)
    except Exception as e:
        return {"layout_problems": [], "flags": ["frame_check_error"],
                "trace": [f"frame_check[{model}]: error — {type(e).__name__}: {e}"]}
    if problems:
        return {"flags": ["manim_layout_issue"], "layout_problems": problems,
                "trace": [f"frame_check[{model}]: FAILED — {problems}"]}
    return {"layout_problems": [], "trace": [f"frame_check[{model}]: ok"]}


## 16 · Terminal nodes + routers + graph assembly

In [ ]:
# === score/withhold/finalize + routers + graph assembly ===
# Reproduces solvi/graph/{nodes.py terminals, routers.py, build.py} using
# langgraph (a dependency, not solvi). viz_type is present on Classification, so
# route_engine works end-to-end.
from langgraph.graph import END, START, StateGraph

HARD_FLAGS = {"VERIFICATION_CONTRADICTS_KEY", "step_missing_reasoning", "invalid_smiles",
              "manim_layout_issue", "manim_render_failed", "narration_failed"}


def score(state):
    cls, plan, ver = state["classification"], state["plan"], state["verification"]
    c = 0.30 * cls.subject_confidence
    c += 0.40 * (1.0 if ver.agrees_with_key else 0.0 if ver.agrees_with_key is False else 0.5)
    complete = all([plan.concept_primer, plan.approach, plan.steps, plan.trap])
    c += 0.15 * (1.0 if complete else 0.0)
    steps_reasoned = bool(plan.steps) and all(s.why for s in plan.steps)
    c += 0.15 * (1.0 if steps_reasoned else 0.0)
    flags = [] if steps_reasoned else ["step_missing_reasoning"]
    return {"confidence": c, "flags": flags, "trace": [f"score: {c:.2f}"]}


def withhold(state):
    return {"status": "requires_review",
            "reason": "confidence below threshold; artifact withheld by policy",
            "flags": ["withheld_low_confidence"],
            "trace": ["withhold: artifact withheld (low confidence)"]}


def finalize(state):
    if state.get("status") in ("requires_review", "unsupported"):
        return {"trace": [f"finalize: {state['status']} (already set)"]}
    hard = HARD_FLAGS & set(state.get("flags", []))
    if "manim_layout_issue" in hard and not state.get("layout_problems"):
        hard.discard("manim_layout_issue")
    status = "requires_review" if hard else "ok"
    return {"status": status, "trace": [f"finalize: {status}"]}


def unsupported(state):
    return {"trace": ["unsupported: terminal"]}


DEFAULT_NODES = {
    "ingest": ingest, "classify": classify, "kc_tag": kc_tag, "solve": solve,
    "verify": verify, "score": score, "gen_manim": generate_manim,
    "gen_rdkit": generate_rdkit, "gen_mol3d": generate_mol3d, "gen_matter": generate_matter,
    "gen_canvas": generate_canvas, "gen_text": gen_text, "frame_check": frame_check,
    "gen_script": gen_script, "script_check": script_check, "gen_audio": gen_audio,
    "audio_check": audio_check, "mux_audio": mux_audio, "withhold": withhold,
    "finalize": finalize, "unsupported": unsupported,
}

# ---- routers (solvi/graph/routers.py) ----
VIZ_TO_ENGINE = {
    VizType.EQUATION_MORPH: Engine.MANIM, VizType.GRAPH_GEOMETRY: Engine.MANIM,
    VizType.RAY_FIELD: Engine.MANIM, VizType.SIMULATED_MOTION: Engine.MATTER,
    VizType.STATIC_DIAGRAM: Engine.CANVAS, VizType.MOLECULE_2D: Engine.RDKIT,
    VizType.MOLECULE_3D: Engine.MOL3D, VizType.NONE: Engine.TEXT,
}
ENGINE_TO_NODE = {
    Engine.MANIM: "gen_manim", Engine.RDKIT: "gen_rdkit", Engine.MOL3D: "gen_mol3d",
    Engine.MATTER: "gen_matter", Engine.CANVAS: "gen_canvas", Engine.TEXT: "gen_text",
}


def choose_engine(cls, cfg):
    if cls.subject_confidence < cfg.routing_confidence_floor:
        return Engine.TEXT
    engine = VIZ_TO_ENGINE[cls.viz_type]
    if engine in cfg.disabled_engines:
        return Engine.TEXT
    return engine


def route_after_ingest(state):
    return "unsupported" if state.get("status") == "unsupported" else "classify"


def route_after_verify(state):
    ver = state["verification"]
    attempts = state.get("solve_attempts", 0)
    if ver.agrees_with_key is False and attempts < _run_config(state).max_solve_attempts:
        return "solve"
    return "score"


def route_engine(state):
    cfg = _run_config(state)
    if state.get("confidence", 0.0) < cfg.min_confidence:
        return "withhold" if cfg.on_low_confidence == "withhold" else "gen_text"
    node = ENGINE_TO_NODE[choose_engine(state["classification"], cfg)]
    if node == "gen_manim" and cfg.narration:
        return "gen_script"
    return node


def route_after_script_check(state):
    if state.get("narration_script") is None:
        return "gen_manim"
    if state.get("narration_problems"):
        return "gen_script"
    return "gen_audio"


def route_after_audio_check(state):
    if state.get("audio_track") is not None and state.get("narration_problems"):
        return "gen_script"
    return "gen_manim"


def route_after_frame_check(state):
    if ("manim_layout_issue" in state.get("flags", []) and state.get("layout_problems")
            and state.get("render_attempts", 0) < _run_config(state).max_render_attempts):
        return "gen_manim"
    return "mux_audio"


# ---- build_graph (solvi/graph/build.py) ----
GENERATOR_NODES = ("gen_manim", "gen_rdkit", "gen_mol3d", "gen_matter", "gen_canvas", "gen_text")


def build_graph(checkpointer=None, node_overrides=None):
    impl = {**DEFAULT_NODES, **(node_overrides or {})}
    g = StateGraph(SolviState)
    for name in ("ingest", "unsupported", "classify", "kc_tag", "solve", "verify", "score"):
        g.add_node(name, impl[name])
    for name in GENERATOR_NODES:
        g.add_node(name, impl[name])
    for name in ("frame_check", "gen_script", "script_check", "gen_audio", "audio_check",
                 "mux_audio", "withhold", "finalize"):
        g.add_node(name, impl[name])

    g.add_edge(START, "ingest")
    g.add_conditional_edges("ingest", route_after_ingest, ["classify", "unsupported"])
    g.add_edge("unsupported", END)
    g.add_edge("classify", "kc_tag")
    g.add_edge("kc_tag", "solve")
    g.add_edge("solve", "verify")
    g.add_conditional_edges("verify", route_after_verify, ["solve", "score"])
    g.add_conditional_edges("score", route_engine, [*GENERATOR_NODES, "gen_script", "withhold"])
    g.add_edge("gen_script", "script_check")
    g.add_conditional_edges("script_check", route_after_script_check,
                            ["gen_script", "gen_audio", "gen_manim"])
    g.add_edge("gen_audio", "audio_check")
    g.add_conditional_edges("audio_check", route_after_audio_check, ["gen_script", "gen_manim"])
    g.add_edge("gen_manim", "frame_check")
    g.add_conditional_edges("frame_check", route_after_frame_check, ["gen_manim", "mux_audio"])
    g.add_edge("mux_audio", "finalize")
    for name in GENERATOR_NODES[1:]:
        g.add_edge(name, "finalize")
    g.add_edge("withhold", "finalize")
    g.add_edge("finalize", END)
    return g.compile(checkpointer=checkpointer)


GRAPH = build_graph()
print("graph compiled:", len(DEFAULT_NODES), "nodes")
print(GRAPH.get_graph().draw_mermaid())


## 17 · Run the pipeline end-to-end

In [ ]:
# === Run the full pipeline end-to-end ===
# Routing presets (what viz_type the classifier tends to pick → engine):
#   circuit / energy-level physics -> static_diagram -> CANVAS  (pure-Python SVG,
#       no CDN/toolchain server-side: a clean HTML artifact every time)
#   maths / calculus              -> equation_morph  -> MANIM   (needs LaTeX+ffmpeg;
#       degrades to text with 'manim_render_failed' if absent)
#   projectile / collision        -> simulated_motion-> MATTER  (browser JS)
#   coordination complex          -> molecule_3d     -> MOL3D
#   organic structure             -> molecule_2d     -> RDKIT
#   statement / assertion         -> none            -> TEXT
question = {
    "id": "sample_circuit",
    "text": "A 6 ohm resistor and a 3 ohm resistor are connected in parallel across "
            "a 12 V battery of negligible internal resistance. What is the total "
            "current drawn from the battery?",
    "options": ["2 A", "4 A", "6 A", "9 A"],
    "answer_key": "6 A",
    "subject_hint": "physics",
    "source": "notebook walkthrough",
}
# Or load any bundled sample:
# question = json.loads((REPO_ROOT / "evals/samples/maths_q.json").read_text())

final = GRAPH.invoke({"raw_input": question, "run_config": RUN_CONFIG,
                      "flags": [], "trace": []})

print("── trace ──")
for line in final.get("trace", []):
    print(" ", line)
print("\n── result ──")
print("  status    :", final.get("status"))
print("  engine    :", getattr(final.get("engine"), "value", None))
print("  artifact  :", final.get("artifact_type"),
      f"({len(final['artifact'])} chars)" if isinstance(final.get("artifact"), str)
      and final.get("artifact_type") == "html" else final.get("artifact"))
print("  confidence:", round(final.get("confidence", 0.0), 2))
print("  flags     :", final.get("flags") or "(none)")
if final.get("reason"):
    print("  reason    :", final["reason"])

# Assemble the public Solution (same mapping as the FastAPI service).
_script = final.get("narration_script")
solution = Solution(
    question_id=final["question"].id if final.get("question") else question["id"],
    status=final.get("status") or "requires_review",
    artifact=final.get("artifact"),
    artifact_type=final.get("artifact_type"),
    engine_used=final.get("engine"),
    plan=final.get("plan"),
    classification=final.get("classification"),
    narration_script="\n".join(b.text for b in _script.beats) if _script else None,
    verification=final.get("verification"),
    confidence=final.get("confidence", 0.0),
    flags=final.get("flags", []),
    trace=final.get("trace", []),
    reason=final.get("reason"),
)
print("\n── Solution (classification + verification) ──")
if solution.classification:
    c = solution.classification
    print(f"  {c.subject.value} / {c.topic} | viz={c.viz_type.value} | "
          f"complexity={c.question_complexity} | chapter={c.chapter_id} | kcs={c.kc_ids}")
if solution.verification:
    v = solution.verification
    print(f"  verify[{v.method}]: ran={v.ran} agrees={v.agrees_with_key} — {v.detail}")


## 18 · Inspect the artifact

In [ ]:
# === Inspect the artifact ===
# HTML engines (text/canvas/matter/mol3d/rdkit) return a self-contained page as a
# string — render it inline. Manim returns an mp4 path (or degrades to HTML).
from IPython.display import HTML, display

art = final.get("artifact")
if final.get("artifact_type") == "html" and isinstance(art, str):
    # Save a copy too, in case you want to open it in a full browser tab.
    out = ARTIFACT_DIR / f"{question['id']}.html"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(art)
    print("wrote", out)
    if any(f in final.get("flags", []) for f in ("manim_render_failed", "invalid_smiles")):
        print("NOTE: an engine degraded to the text artifact — see flags above.")
    display(HTML(art))
elif art:
    print("video artifact at:", art)
else:
    print("no artifact (status:", final.get("status"), "reason:", final.get("reason"), ")")
